<a href="https://colab.research.google.com/github/MicheleQGF/ProjectAllocationBudgetSynthetic/blob/main/FullProjectAllocationBudget(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from pathlib import Path

# Find the path the notebook is using
user_home = Path.home()
print("Tu carpeta de usuario real es:", user_home)

Tu carpeta de usuario real es: C:\Users\PC


In [ ]:
# Change the path
nueva_ruta = r"C:\Users\PC\Documents\TAWOS\tawos_data\FullProjectAllocationBudgetDatasets"

os.chdir(nueva_ruta)

# Verify it changed
print("Ahora estás en:", os.getcwd())

Ahora estás en: C:\Users\PC\Documents\TAWOS\tawos_data\FullProjectAllocationBudgetDatasets


# 1- Datasets Exploration
** the nulls in the `projects_datasets` were reviewd on the `evm_sprint`construction**

In [ ]:
import pandas as pd
import numpy as np
# Load the projects_dataset which have real TAWOS projects data
prod = pd.read_csv('projects_dataset.csv', low_memory=False)
prod.head()

,ID,Issue_Key,Title,Type,Priority,Status,Resolution,Creation_Date,Estimation_Date,Resolution_Date,...,Resolution_Month,Is_Feature,Rolling_HPP,Planned_Hours,Planned_Value_USD,Original_SP,SP_Truly_Revised,PV_Frozen,Created_Year,Tier_Label
0,37998,TIDOC-3403,"""Documentation: MacOS icons missing""",Improvement,NaN,Open,NaN,2020-09-29 12:46:30,NaN,NaN,...,NaN,False,28.34868,NaN,NaN,NaN,False,NaN,2020,No tier (assumed rate)
1,37999,TIDOC-3401,"""Use actual constants instead of wildcard cons...",Bug,High,Closed,Fixed,2020-09-10 21:00:07,NaN,2020-09-10 23:19:14,...,2020-09-01,False,28.34868,NaN,NaN,NaN,False,NaN,2020,No tier (assumed rate)
2,38000,TIDOC-3400,"""Webpack: Getting started guide""",Story,NaN,Closed,Done,2020-08-10 14:28:40,NaN,2020-10-01 10:26:53,...,2020-10-01,False,28.34868,NaN,NaN,NaN,False,NaN,2020,Junior
3,38001,TIDOC-3399,"""Remove references to """"support@appcelerator.c...",Story,NaN,Open,NaN,2020-05-21 21:17:51,NaN,NaN,...,NaN,False,28.34868,NaN,NaN,NaN,False,NaN,2020,No tier (assumed rate)
4,38002,TIDOC-3398,"""Document iOS showInForeground push property""",Story,NaN,Open,NaN,2020-03-26 18:40:04,NaN,NaN,...,NaN,False,28.34868,NaN,NaN,NaN,False,NaN,2020,No tier (assumed rate)


In [ ]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 20854 entries, 0 to 20853
Data columns (total 67 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       20854 non-null  int64  
 1   Issue_Key                20854 non-null  str    
 2   Title                    20854 non-null  str    
 3   Type                     20854 non-null  str    
 4   Priority                 20205 non-null  str    
 5   Status                   20854 non-null  str    
 6   Resolution               19219 non-null  str    
 7   Creation_Date            20854 non-null  str    
 8   Estimation_Date          8873 non-null   str    
 9   Resolution_Date          19218 non-null  str    
 10  Last_Updated             20854 non-null  str    
 11  Story_Point              8873 non-null   float64
 12  Timespent                853 non-null    float64
 13  In_Progress_Minutes      20854 non-null  float64
 14  Total_Effort_Minutes     20854 no

In [ ]:
prod['Assignee_ID_Final'].value_counts()

Assignee_ID_Final
-1         1604
 165193    1402
 165280    1234
 9008      1159
 165168     867
           ... 
 165299       1
 165763       1
 165814       1
 165660       1
 165282       1
Name: count, Length: 273, dtype: int64

In [ ]:
date_cols = [c for c in prod.columns
             if any(k in c for k in ['Date', 'Last_Updated', 'Sprint_Start', 'Sprint_End', 'Sprint_Complete', 'Estimation_Month', 'Resolution_Month'])]
print(date_cols)

for c in date_cols:
    prod[c] = pd.to_datetime(prod[c], errors='coerce')   # bad values become NaT instead of erroring

print(prod[date_cols].dtypes)
print(prod[date_cols].isna().sum())   # should match the null counts you saw before

['Creation_Date', 'Estimation_Date', 'Resolution_Date', 'Last_Updated', 'Sprint_Start', 'Sprint_End', 'Estimation_Month', 'Resolution_Month']
Creation_Date       datetime64[us]
Estimation_Date     datetime64[us]
Resolution_Date     datetime64[us]
Last_Updated        datetime64[us]
Sprint_Start        datetime64[us]
Sprint_End          datetime64[us]
Estimation_Month    datetime64[us]
Resolution_Month    datetime64[us]
dtype: object
Creation_Date           0
Estimation_Date     11981
Resolution_Date      1636
Last_Updated            0
Sprint_Start        14950
Sprint_End          14950
Estimation_Month    11981
Resolution_Month     1636
dtype: int64


In [ ]:
# Load the team_profiles which have synthetic roles/measures
tp = pd.read_csv('team_profiles_v1.csv', low_memory=False)
tp.head()

,Assignee_ID,Total_Issues,First_Activity,Last_Activity,Avg_Resolution_Minutes,Tenure_Days,Group,Avg_Story_Point,Pct_High_Priority,Tenure_Score,...,Productivity_Norm,Training_Norm,Adaptability_Norm,Collaboration_Norm,Efficiency_Norm,HRA_Seniority_Score,Performance_Score,Experience_Score,Growth_Score,Reliability_Score
0,8850,1,2016-04-27 14:26:38,2016-04-27 14:26:38,7509.000000,0,One-off,NaN,NaN,0.000000,...,0.436700,0.238984,0.592838,0.159473,0.099036,0.083000,0.330766,0.120477,0.281709,0.430813
1,8851,10,2017-05-22 17:37:51,2020-08-10 14:28:40,66351.800000,1175,Core,7.000000,14.285714,0.249522,...,0.700245,0.583677,0.665908,0.697467,0.361087,0.239949,0.546583,0.176732,0.430941,0.743731
2,8852,14,2016-07-01 23:17:36,2020-03-02 16:10:15,79612.285714,1339,Core,2.333333,8.333333,0.284349,...,0.618023,0.380725,0.494515,0.619250,0.161262,0.229055,0.544300,0.286618,0.404459,0.568060
3,8855,1,2016-03-26 10:17:22,2016-03-26 10:17:22,377618.000000,0,One-off,NaN,NaN,0.000000,...,0.217533,0.481875,0.680397,0.649645,0.185802,0.083422,0.237968,0.118021,0.414771,0.702950
4,8857,1,2019-12-10 13:32:54,2019-12-10 13:32:54,0.000000,0,One-off,NaN,NaN,0.000000,...,0.378710,0.391144,0.153591,0.522796,0.254163,0.089399,0.423241,0.166446,0.229250,0.673702


In [ ]:
tp.info()

<class 'pandas.DataFrame'>
RangeIndex: 182 entries, 0 to 181
Data columns (total 70 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Assignee_ID                   182 non-null    int64  
 1   Total_Issues                  182 non-null    int64  
 2   First_Activity                182 non-null    str    
 3   Last_Activity                 182 non-null    str    
 4   Avg_Resolution_Minutes        182 non-null    float64
 5   Tenure_Days                   182 non-null    int64  
 6   Group                         182 non-null    str    
 7   Avg_Story_Point               73 non-null     float64
 8   Pct_High_Priority             129 non-null    float64
 9   Tenure_Score                  182 non-null    float64
 10  Volume_Score                  182 non-null    float64
 11  Complexity_Score              182 non-null    float64
 12  Priority_Score                182 non-null    float64
 13  Seniority_Score 

In [ ]:
date_colsb = [c for c in tp.columns
             if any(k in c for k in ['First_Activity', 'Last_Activity'])]
print(date_colsb)

for c in date_colsb:
    tp[c] = pd.to_datetime(tp[c], errors='coerce')   # bad values become NaT instead of erroring

print(tp[date_colsb].dtypes)
print(tp[date_colsb].isna().sum())   # should match the null counts you saw before

['First_Activity', 'Last_Activity']
First_Activity    datetime64[us]
Last_Activity     datetime64[us]
dtype: object
First_Activity    0
Last_Activity     0
dtype: int64


In [ ]:
# Load the evm_sprint which have real TAWOS projects data
evm = pd.read_csv('evm_sprint.csv', low_memory=False)
evm.head()

,Project_Name,Sprint_ID,Sprint_Start,Sprint_End,Issues,Scope_Start_Pts,Scope_End_Pts,Scope_End_Pts_ND,EV_Pts,AC_Done_Hours,...,AC_All_USD,BAC_Total_USD,BAC_USD,AC_USD,PV_USD,EV_USD,CPI_USD,Pct_Budget_Used,EAC_USD,VAC_USD
0,Appcelerator Studio,4668.0,2012-09-27 05:26:47,2012-10-08 20:05:00,1,0.0,0.0,0.0,0.0,0.000000,...,7.590044e+05,7.719452e+06,5.908544e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Appcelerator Studio,4667.0,2013-07-02 20:29:00,2013-07-15 20:29:00,13,57.0,57.0,57.0,52.0,122.266667,...,1.786736e+06,7.719452e+06,5.908544e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Appcelerator Studio,4666.0,2013-07-16 15:31:18,2013-07-29 15:31:18,18,57.0,70.0,70.0,57.0,222.750000,...,1.802546e+06,7.719452e+06,5.908544e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Appcelerator Studio,4665.0,2013-07-30 23:54:12,2013-08-12 23:54:12,16,75.0,62.0,75.0,54.0,530.583333,...,1.838409e+06,7.719452e+06,5.908544e+06,30251.617667,31227.242799,59940.145166,1.981386,0.238153,NaN,NaN
4,Appcelerator Studio,4664.0,2013-08-14 00:48:00,2013-08-24 00:48:00,16,84.0,92.0,92.0,136.0,424.333333,...,1.902559e+06,7.719452e+06,5.908544e+06,94401.158000,57740.039359,210900.510768,2.234088,0.246463,NaN,NaN


In [ ]:
evm.info()

<class 'pandas.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 46 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Project_Name         549 non-null    str    
 1   Sprint_ID            549 non-null    float64
 2   Sprint_Start         549 non-null    str    
 3   Sprint_End           549 non-null    str    
 4   Issues               549 non-null    int64  
 5   Scope_Start_Pts      549 non-null    float64
 6   Scope_End_Pts        549 non-null    float64
 7   Scope_End_Pts_ND     549 non-null    float64
 8   EV_Pts               549 non-null    float64
 9   AC_Done_Hours        549 non-null    float64
 10  AC_Hours             549 non-null    float64
 11  Late_Descope_Pts     549 non-null    float64
 12  Prev_Done_Hours      546 non-null    float64
 13  Prev_EV_Pts          546 non-null    float64
 14  Baseline_HPP         530 non-null    float64
 15  Actual_HPP           468 non-null    float64
 16  P

In [ ]:

   print('Final_EV_Pts' in evm.columns, evm.shape)
   print([c for c in evm.columns if 'PV' in c or 'SPI' in c or 'Final' in c])

True (549, 46)
['Final_EV_Pts', 'PV_Pts_Time', 'SPI', 'PV_USD']


In [ ]:
date_colsc = [c for c in evm.columns
             if any(k in c for k in ['Sprint_Start', 'Sprint_End'])]
print(date_colsc)

for c in date_colsc:
    evm[c] = pd.to_datetime(evm[c], errors='coerce')   # bad values become NaT instead of erroring

print(evm[date_colsc].dtypes)
print(evm[date_colsc].isna().sum())   # should match the null counts you saw before

['Sprint_Start', 'Sprint_End']
Sprint_Start    datetime64[us]
Sprint_End      datetime64[us]
dtype: object
Sprint_Start    0
Sprint_End      0
dtype: int64


In [ ]:
# Load the project_budgets dataset
pb = pd.read_csv('project_budgets.csv', low_memory=False)
pb.head()

,Project_Name,Hours,Actual_Cost_USD,Uncapped_Cost_USD,Pct_Cost_Assumed_Rate,Scenario,Factor,Annual_Budget_USD,Simulated_Budget_USD,Variance_USD,Variance_%,Budget_Status
0,Appcelerator Studio,185927.816667,8.062786e+06,1.393365e+07,2.5,On budget (fixed annual),NaN,800000.0,7.719452e+06,-3.433343e+05,-4.4,On budget (within 5%)
1,Mule,562968.150000,2.204802e+07,4.745110e+07,1.9,Under budget (comfortable),1.20,NaN,2.645762e+07,4.409603e+06,16.7,Under budget
2,Titanium Mobile Platform,312799.850000,1.092503e+07,3.131084e+07,1.8,Over budget (tight),0.85,NaN,9.286272e+06,-1.638754e+06,-17.6,Over budget


In [ ]:
pb.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Project_Name           3 non-null      str    
 1   Hours                  3 non-null      float64
 2   Actual_Cost_USD        3 non-null      float64
 3   Uncapped_Cost_USD      3 non-null      float64
 4   Pct_Cost_Assumed_Rate  3 non-null      float64
 5   Scenario               3 non-null      str    
 6   Factor                 2 non-null      float64
 7   Annual_Budget_USD      1 non-null      float64
 8   Simulated_Budget_USD   3 non-null      float64
 9   Variance_USD           3 non-null      float64
 10  Variance_%             3 non-null      float64
 11  Budget_Status          3 non-null      str    
dtypes: float64(9), str(3)
memory usage: 420.0 bytes


In [ ]:
# Load the cost_monthly dataset
cm = pd.read_csv('cost_monthly.csv', low_memory=False)
cm.head()

,Project_Name,Month,Capped_Cost_USD,Cum_Actual_Cost_USD
0,Appcelerator Studio,2011-03-01,31956.625000,31956.625000
1,Appcelerator Studio,2011-04-01,11107.818833,43064.443833
2,Appcelerator Studio,2011-05-01,5012.394833,48076.838667
3,Appcelerator Studio,2011-06-01,3.300000,48080.138667
4,Appcelerator Studio,2011-07-01,2076.355667,50156.494333


In [ ]:
cm['Project_Name'].value_counts()

Project_Name
Mule                        194
Titanium Mobile Platform    107
Appcelerator Studio         106
Name: count, dtype: int64

In [ ]:
cm.info()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Project_Name         407 non-null    str    
 1   Month                407 non-null    str    
 2   Capped_Cost_USD      407 non-null    float64
 3   Cum_Actual_Cost_USD  407 non-null    float64
dtypes: float64(2), str(2)
memory usage: 12.8 KB


# 2- Overview calculations: financials, EVM, effort and staffing #

**What this section answers:** how much each project cost against its budget, how delivery performed (EVM), where the effort went by type of work and by seniority tier, and how concentrated that effort was.

**Read this first.** The data is a mix of real and simulated parts:
- *Real (TAWOS Jira):* issues, story points, dates, change history, effort time.
- *Synthetic:* the HRA employee profiles (matched to the contributors), the seniority tiers built from them, the hourly rates and the budgets.
- Hours are **elapsed ticket time** and each issue is capped at 720 h. Costs are therefore an upper-bound estimate, not real spend.

## 2.1 Financials: budget vs cost

Budgets are simulated: one total per project, built to give three different situations (see the budget notebook). Cost is capped hours × tier rate. Variance = budget − cost, so **positive means under budget**.

| Project | Scenario | Budget | Cost | Variance | % of budget used |
|---|---|---|---|---|---|
| Appcelerator Studio | On budget (fixed annual) | $7.72M | $8.06M | −$0.34M (−4.4%) | 104.4% |
| Mule | Under budget | $26.46M | $22.05M | +$4.41M (+16.7%) | 83.3% |
| Titanium Mobile Platform | Over budget | $9.29M | $10.93M | −$1.64M (−17.6%) | 117.6% |

- The figures were recalculated from the issue-level data in Python and match the budget file exactly. This is the reference for the Tableau cards.
- Cumulative cost against the budget: Titanium crossed its budget in August 2018. Mule never did.
- The monthly cost series places each issue in its resolution month. Issues with no resolution date (open) are not in it, which is why the monthly line ends slightly below the card total: $355K for Appcelerator, $94K for Mule and $37K for Titanium. Including them, Appcelerator is above its budget.
- The budget is a single total per project, not a plan by month, so the cumulative chart compares cost against a flat line.

In [ ]:
w = prod[prod['Capped_Hours'] > 0].copy()
w['Year'] = w['Resolution_Date'].dt.year

cards = (prod.groupby('Project_Name')
             .agg(Issues=('ID', 'count'),
                  Hours=('Capped_Hours', 'sum'),
                  Cost_Capped=('Capped_Cost_USD', 'sum'),
                  Cost_Uncapped=('Actual_Cost_USD', 'sum'),
                  Points_Done=('Story_Point', 'sum'))
             .reset_index())

assumed = (prod[prod['Rate_Source'].str.contains('ssum', case=False, na=False)]
             .groupby('Project_Name')['Capped_Cost_USD'].sum().rename('Cost_Assumed_Rate'))
cards = cards.merge(assumed, on='Project_Name', how='left').fillna({'Cost_Assumed_Rate': 0})
cards['Pct_Assumed_Rate'] = cards['Cost_Assumed_Rate'] / cards['Cost_Capped']
cards['Pct_Cut_by_Cap']   = 1 - cards['Cost_Capped'] / cards['Cost_Uncapped']
cards['Cost_per_Issue']   = cards['Cost_Capped'] / cards['Issues']

cost_year = (w.dropna(subset=['Year']).groupby(['Project_Name', 'Year'])['Capped_Cost_USD'].sum().reset_index())
cost_year['Cum_Cost'] = cost_year.groupby('Project_Name')['Capped_Cost_USD'].cumsum()

pd.options.display.float_format = '{:,.2f}'.format
print(cards.to_string(index=False))
print(cost_year.groupby('Project_Name').tail(1))   # last year's Cum_Cost should equal Cost_Capped, apart from issues with no resolution date

            Project_Name  Issues      Hours   Cost_Capped  Cost_Uncapped  Points_Done  Cost_Assumed_Rate  Pct_Assumed_Rate  Pct_Cut_by_Cap  Cost_per_Issue
     Appcelerator Studio    5979 185,927.82  8,062,786.34  13,933,647.21    18,658.00         205,080.81              0.03            0.42        1,348.52
                    Mule   11816 562,968.15 22,048,016.91  47,451,102.43    18,593.00         427,195.40              0.02            0.54        1,865.95
Titanium Mobile Platform    3059 312,799.85 10,925,025.39  31,310,836.58     5,915.50         195,947.37              0.02            0.65        3,571.44
                Project_Name     Year  Capped_Cost_USD      Cum_Cost
9        Appcelerator Studio 2,020.00       135,033.80  7,707,728.36
26                      Mule 2,020.00        42,159.85 21,954,117.53
36  Titanium Mobile Platform 2,020.00        14,431.67 10,887,987.11


In [ ]:
from_issues = (prod.groupby('Project_Name')
                   .agg(Hours_py=('Capped_Hours', 'sum'),
                        Capped_Cost_py=('Capped_Cost_USD', 'sum'),
                        Uncapped_Cost_py=('Actual_Cost_USD', 'sum')))   # in prod this column is the uncapped one

cards = pb.set_index('Project_Name').join(from_issues)
cards['Budget']   = cards['Simulated_Budget_USD']
cards['Variance_py']     = cards['Budget'] - cards['Capped_Cost_py']       # positive = under budget
cards['Variance_pct_py'] = cards['Variance_py'] / cards['Budget']
cards['Pct_Consumed']    = cards['Capped_Cost_py'] / cards['Budget']

pd.options.display.float_format = '{:,.3f}'.format
print(cards[['Scenario', 'Budget', 'Capped_Cost_py', 'Actual_Cost_USD',
             'Variance_py', 'Variance_USD', 'Variance_pct_py', 'Variance_%',
             'Pct_Consumed', 'Budget_Status']].T.to_string())

# reconciliation: all of these should be about 0
print((cards['Actual_Cost_USD'] - cards['Capped_Cost_py']).round(2).to_dict())
print((cards['Uncapped_Cost_USD'] - cards['Uncapped_Cost_py']).round(2).to_dict())
print((cards['Hours'] - cards['Hours_py']).round(2).to_dict())

Project_Name          Appcelerator Studio                        Mule Titanium Mobile Platform
Scenario         On budget (fixed annual)  Under budget (comfortable)      Over budget (tight)
Budget                      7,719,452.055              26,457,620.288            9,286,271.585
Capped_Cost_py              8,062,786.343              22,048,016.907           10,925,025.395
Actual_Cost_USD             8,062,786.343              22,048,016.907           10,925,025.395
Variance_py                  -343,334.288               4,409,603.381           -1,638,753.809
Variance_USD                 -343,334.288               4,409,603.381           -1,638,753.809
Variance_pct_py                    -0.044                       0.167                   -0.176
Variance_%                         -4.400                      16.700                  -17.600
Pct_Consumed                        1.044                       0.833                    1.176
Budget_Status       On budget (within 5%)         

In [ ]:
cm['Month'] = pd.to_datetime(cm['Month'], errors='coerce')
cm = cm.sort_values(['Project_Name', 'Month']).reset_index(drop=True)

# recompute the cumulative cost to check it against the file
cm['Cum_py'] = cm.groupby('Project_Name')['Capped_Cost_USD'].cumsum()
print('max difference vs file:', (cm['Cum_py'] - cm['Cum_Actual_Cost_USD']).abs().max().round(2))

cm = cm.merge(pb[['Project_Name', 'Simulated_Budget_USD']], on='Project_Name', how='left')
cm['Pct_of_Budget'] = cm['Cum_py'] / cm['Simulated_Budget_USD']

# month when the budget is exceeded (NaT = never)
first_over = (cm[cm['Pct_of_Budget'] >= 1].groupby('Project_Name')['Month'].min()
                .reindex(pb['Project_Name']))
print(first_over)

# last month per project: total cost should be close to the cards above
print(cm.groupby('Project_Name').tail(1)[['Project_Name', 'Month', 'Cum_py', 'Simulated_Budget_USD', 'Pct_of_Budget']])

max difference vs file: 0.0
Project_Name
Appcelerator Studio               NaT
Mule                              NaT
Titanium Mobile Platform   2018-08-01
Name: Month, dtype: datetime64[us]
                 Project_Name      Month         Cum_py  Simulated_Budget_USD  \
105       Appcelerator Studio 2020-05-01  7,707,728.359         7,719,452.055   
299                      Mule 2020-09-01 21,954,117.530        26,457,620.288   
406  Titanium Mobile Platform 2020-10-01 10,887,987.111         9,286,271.585   

     Pct_of_Budget  
105          0.998  
299          0.830  
406          1.172  


In [ ]:
open_cost = (prod[prod['Resolution_Date'].isna()]
               .groupby('Project_Name')['Capped_Cost_USD'].sum())
print(open_cost.round(0))
print((cards['Capped_Cost_py'] - cm.groupby('Project_Name')['Capped_Cost_USD'].sum()).round(0))   # gap between cards and monthly

Project_Name
Appcelerator Studio        355,058.000
Mule                        93,899.000
Titanium Mobile Platform    37,038.000
Name: Capped_Cost_USD, dtype: float64
Project_Name
Appcelerator Studio        355,058.000
Mule                        93,899.000
Titanium Mobile Platform    37,038.000
dtype: float64


## 2.2 Earned value (EVM)

### Recap: the per-sprint calculation

| Measure | How it is calculated |
|---|---|
| **EV** (points) | Points of the issues completed in each sprint (Fixed / Done / Completed), with their value on the day they were resolved |
| **Baseline** | Hours per point from all previous sprints (at least 100 completed points before a sprint is priced) |
| **AC** (hours) | Capped hours (720 h per issue at most) of the issues finished in the sprint |
| **CPI** (hours) | Cumulative earned hours ÷ cumulative actual hours |
| **Pct_Done** | Cumulative earned points ÷ the final total of earned points |
| **Pct_Time** | Share of the measured timeline that has elapsed at the sprint end |
| **SPI** | Pct_Done ÷ Pct_Time (blank in the first 5 % of the timeline) |
| **Completion** | Cumulative earned points ÷ cumulative points the sprints held |

### The budget version

| Measure | Calculation |
|---|---|
| **BAC (total)** | `Simulated_Budget_USD` of the project |
| **BAC (measured window)** | BAC × (cost from the first priced sprint ÷ total cost): the budget share that belongs to the period the sprints measure |
| **EV** | BAC (window) × Pct_Done |
| **PV** | BAC (window) × Pct_Time |
| **AC** | Cumulative cost (capped hours × hourly rate) of every resolved issue in the project, counted from the first priced sprint |
| **CPI** | EV ÷ AC |
| **Pct_Budget_Used** | Whole-project cost to date ÷ total BAC |
| **EAC / VAC** | Forecast total cost (cost before the window + BAC window ÷ CPI), and total BAC minus that forecast. Blank until 10 % of the points are delivered |

**Why a measured window.** Sprints, and so earned value, only exist from the first priced sprint, but issues were being resolved and paid for before that. Counting that earlier cost against an EV that starts at zero produces absurd early values (Mule's CPI near 0). So the cost clock starts at the first priced sprint, like the hours-based CPI, and the budget is scaled to the same window in proportion to cost. The final CPI is unaffected (BAC ÷ total cost).

**How the budgets were built.** They are a factor of each project's own total cost: Titanium "tight" (85 %), Mule "comfortable" (120 %) and Appcelerator a fixed annual amount. The **final** CPI therefore follows from that choice and is not a finding. What is informative is the path: when a project went from under to over budget and how fast.

### Why SPI is not recalculated with the budget

SPI = EV ÷ PV. With the budget, EV = BAC × Pct_Done and PV = BAC × Pct_Time, so SPI = Pct_Done ÷ Pct_Time. BAC cancels out: a budget says how much work should cost, not when it should be done. Only the cost measures change, and the dashboards keep a single SPI column.

### There is no real planned value

A true SPI compares progress with a plan agreed before the work started. TAWOS has no schedule, release plan or phased budget, so no SPI here can say whether a project was ahead of or behind a real schedule. The data supports two readings of delivery:

| View | Question it answers | Limits |
|---|---|---|
| **Pace SPI** (Pct_Done ÷ Pct_Time) | Was delivery faster or slower than an even pace over time? | The plan is a straight line, the final total is known in hindsight, and scope kept growing. Above 1 does not mean "ahead of schedule", and it always ends at 1.0 |
| **Completion** | Of everything the sprints held, how much was delivered? | Says nothing about time, and counts work added after the sprint started |

A third option, not calculated, would be a **commitment SPI**: points delivered that were already in scope at sprint start ÷ points in scope at sprint start. It is the closest the data gets to a plan, but it measures how reliable the sprint commitments were, not schedule progress.

### What goes to the dashboards

- **Cost:** BAC, AC, EV, CPI, Pct_Budget_Used, EAC and VAC per sprint, filtered to Sprint Status = Measurable.
- **Schedule:** one SPI column, with Pct_Done and Pct_Time so the pace can be seen directly.
- **Reading guide:** SPI is a pace measure against an assumed plan, and final cost figures follow from how the budgets were simulated.
- **Early values are unstable.** CPI is a cumulative ratio, so in the first sprints after a project becomes measurable the cost and earned value are both tiny and the ratio swings (for example Titanium's first CPI points, around 2.5 in hours and 7 in USD). Read CPI from roughly the point where 10 % of the final points have been delivered. SPI is hidden in the first 5 % of the timeline for the same reason. These early points are kept in the data, and the dashboards fix the axis range so they do not flatten the rest of the line.

## 2.3 Burndown & Burnup

Remaining points = cumulative scope − cumulative completed points, per sprint. This agrees exactly with % complete in the EVM table.

- Appcelerator ends with about 180 points remaining (about 3% of the scope).
- Mule (−266) and Titanium (−326) end with **negative** remaining points, meaning more points were completed than the recorded scope. This matches their % complete above 100% (1.02 and 1.09) and comes from estimates added or changed after the work was done. It is kept in the chart on purpose, as a real feature of the tracking data.

In [ ]:
#burndown
evm['Sprint_Start'] = pd.to_datetime(evm['Sprint_Start'], errors='coerce')
evm['Sprint_End']   = pd.to_datetime(evm['Sprint_End'], errors='coerce')
evm = evm.sort_values(['Project_Name', 'Sprint_End']).reset_index(drop=True)

bd = evm[['Project_Name', 'Sprint_ID', 'Sprint_End', 'Year', 'Sprint_Status',
          'Cum_BAC_Pts', 'Cum_BAC_Pts_ND', 'Cum_EV_Pts']].copy()
bd['Remaining_Pts']    = bd['Cum_BAC_Pts'] - bd['Cum_EV_Pts']
bd['Remaining_Pts_ND'] = bd['Cum_BAC_Pts_ND'] - bd['Cum_EV_Pts']   # without descoping closed-without-delivery work

# sanity check: remaining should agree with Pct_Complete (where it exists)
chk = evm.loc[evm['Pct_Complete'].notna()].copy()
chk['pct_from_pts'] = chk['Cum_EV_Pts'] / chk['Cum_BAC_Pts']
print((chk['pct_from_pts'] - chk['Pct_Complete']).abs().describe().round(3))
print(bd.groupby('Project_Name').tail(1)[['Project_Name', 'Sprint_End', 'Cum_BAC_Pts', 'Cum_EV_Pts', 'Remaining_Pts']])

count   530.000
mean      0.000
std       0.000
min       0.000
25%       0.000
50%       0.000
75%       0.000
max       0.000
dtype: float64
                 Project_Name          Sprint_End  Cum_BAC_Pts  Cum_EV_Pts  \
143       Appcelerator Studio 2020-05-22 16:37:00    5,503.000   5,323.000   
386                      Mule 2020-07-15 14:53:00   12,273.000  12,539.000   
548  Titanium Mobile Platform 2019-10-04 21:00:00    3,634.000   3,960.000   

     Remaining_Pts  
143        180.000  
386       -266.000  
548       -326.000  


### Why remaining points can be negative (Titanium)

In the burndown, remaining work = cumulative scope − cumulative points delivered. In Titanium the line goes below zero and ends a few hundred points under it, which looks like "negative work". It is not: it means more points were counted as delivered than the sprints recorded as scope.

**Cause.** The two sides are measured on different dates:

- **Scope** is counted at the **end of each sprint**, with the points each issue held on that date.
- **Delivered points** are counted when the issue is **resolved**, with the points it held that day.

About 10 % of Titanium's completed issues gained points between the end of their sprint and their resolution (their estimate was raised after the sprint closed). Those extra points are counted as delivered but were never part of any recorded scope. Over the project this adds up to 3,946 delivered points against a scope of 3,629 (+8.7 %). The effect is visible from the first year (completion above 1.0 from 2014) and stays between 1.03 and 1.09.

**Other projects.** Mule delivers +2.0 % more points than its recorded scope and Appcelerator −1.5 %, so their remaining work stays close to zero at the end.

**How to read it.** Negative remaining work means "no work left and the recorded scope is slightly understated". It does not mean the team did extra work. Remaining work is left unfloored in the charts so the effect stays visible, and the burn-up view (scope and delivered as two lines) shows the same thing directly: the delivered line ends above the scope line.

**What would fix it.** Counting delivered points with the value they had at the end of the sprint (instead of at resolution) would bring both sides to the same date. It was not done because the points at resolution are the best estimate of the work that was actually delivered.

In [ ]:
# Burn-up: scope vs points delivered ----------
P = evm['Priced']
evm['Scope_Pts']          = evm['Cum_BAC_Pts'].where(P)                  # cumulative scope, descoped issues removed
evm['Delivered_Pts']      = evm['Cum_EV_Pts'].where(P)                   # cumulative points delivered
evm['Remaining_Pts']      = evm['Scope_Pts'] - evm['Delivered_Pts']      # negative = delivered more than recorded scope
evm['Delivered_vs_Scope'] = evm['Delivered_Pts'] / evm['Scope_Pts']      # same as Pct_Complete
evm['Ideal_Remaining_Pts'] = (evm.groupby('Project_Name')['Cum_BAC_Pts'].transform('max') * (1 - evm['Pct_Time'])).where(P)

ok = evm[evm['Priced']]
print("Burn-up at the end of each year:")
print(ok.groupby(['Project_Name', 'Year'])[['Scope_Pts', 'Delivered_Pts', 'Remaining_Pts', 'Delivered_vs_Scope']].last().round(2), "\n")

last = ok.groupby('Project_Name').tail(1).set_index('Project_Name')
print("Final burn-up (compare with the burndown check: about +180 / -266 / -326 remaining):")
print(last[['Scope_Pts', 'Delivered_Pts', 'Remaining_Pts', 'Delivered_vs_Scope']].round(2))

Burn-up at the end of each year:
                               Scope_Pts  Delivered_Pts  Remaining_Pts  \
Project_Name             Year                                            
Appcelerator Studio      2013  1,171.000      1,119.000         52.000   
                         2014  3,086.000      2,934.000        152.000   
                         2015  3,988.000      3,853.000        135.000   
                         2016  4,540.000      4,319.000        221.000   
                         2017  5,031.000      4,880.000        151.000   
                         2018  5,416.000      5,231.000        185.000   
                         2019  5,448.000      5,277.000        171.000   
                         2020  5,503.000      5,323.000        180.000   
Mule                     2013     75.000         68.000          7.000   
                         2014  1,591.000      1,497.000         94.000   
                         2015  2,493.000      2,476.000         17.000   
     

### Burn-up: scope vs points delivered

The burn-up chart draws two lines per project, from the sprint table: **scope** (cumulative points the sprints hold, with descoped issues removed) and **delivered** (cumulative points of the issues completed). The gap between them is the remaining work.

#### Why a burn-up and not only a burndown

A burndown has one line (remaining = scope − delivered), so it mixes two things: how much the team delivered and how much work was added. Scope grew 35–49 % after the sprints started, so a flat burndown could mean a slow team or a growing scope, and the chart cannot say which. The burn-up keeps them apart: the delivered line shows progress, and the scope line shows growth. The burndown is kept as a smaller companion chart for "how much is left".

#### What the chart shows

| Project | Delivered vs scope at the end | Reading |
|---|---|---|
| Appcelerator Studio | 97 % | The delivered line stays just under scope the whole way. The team kept up with the scope and never closed the last 3 % (about 180 points) |
| Mule | 102 % | Delivered lags scope until about mid-2018, then catches up and finishes just above it. The straight diagonal near the end is a stretch with few sprint end dates where roughly 1,000 points were delivered |
| Titanium Mobile Platform | 109 % | Delivered is above scope from 2015 and ends above it. See the note on negative remaining work |

#### Why SPI does not show the missing 3 % in Appcelerator

The pace SPI compares progress with what the project **eventually delivered** (Pct_Done = points delivered so far ÷ final points delivered), not with the scope. It therefore reaches 100 % at the end by construction, even when part of the scope was never finished. SPI answers "was the pace even over time?", not "was the scope completed?". The completion measure (delivered ÷ scope: 0.97, 1.02 and 1.09) and the burn-up gap answer the second question, so both are shown next to SPI.

#### Limits

- The chart covers the points that belong to sprints. Appcelerator's sprints hold only about 38 % of its pointed issues (Mule 81.5 %, Titanium 76 %), so the chart shows how well the sprints delivered their own scope, not the whole project. For the full cost and hours picture, see the overview.
- Delivered points are counted with their value on the day of resolution and scope with their value at each sprint end, which is why Titanium ends above 100 % (see the note on negative remaining work).

## 2.4 Hours by type of task

Each project uses its own Jira issue types, so bars cannot be compared one to one across projects.

- **Appcelerator:** Bug 49% of hours, then Improvement 23% and Story 17%.
- **Mule:** Task 34%, Enhancement Request 32% and Bug 31%. It has no New Feature or Improvement type.
- **Titanium:** Bug 35%, New Feature 28% and Story 16%.
- Overall, bugs are the largest block of effort (about 375K hours, roughly a third of the total).
- Mule's hours per issue (about 147–163 for its main types) are around double Appcelerator's. That reflects tickets staying open longer (elapsed time), not necessarily more work.
- For a cross-project comparison, `Type_Group` joins the types into Corrective (bugs), New functionality, Improvement and Tasks/technical. **Assumption:** Enhancement Request counts as new functionality.

In [ ]:
# 1) group the types of tasks so the projects can be compared
#    (assumption: 'Enhancement Request' counts as new functionality)
type_map = {
    'Bug': 'Corrective (bugs)',
    'New Feature': 'New functionality', 'Enhancement Request': 'New functionality',
    'Story': 'New functionality', 'Epic': 'New functionality',
    'Improvement': 'Improvement',
    'Task': 'Tasks / technical', 'Sub-task': 'Tasks / technical', 'Technical task': 'Tasks / technical',
}
prod['Type_Group'] = prod['Type'].map(type_map)
print(prod['Type_Group'].value_counts(dropna=False), '\n')   # NaN here = a type missing from the map

# 2) issues with effort only
w = prod[prod['Capped_Hours'] > 0].copy()

# 3) hours per project, by raw type and by group
def hours_table(df, by):
    t = (df.groupby(['Project_Name', by])
           .agg(Issues=('ID', 'count'), Hours=('Capped_Hours', 'sum'), Cost_USD=('Capped_Cost_USD', 'sum'))
           .reset_index())
    t['Share_Hours'] = t['Hours'] / t.groupby('Project_Name')['Hours'].transform('sum')
    t['Hours_per_Issue'] = t['Hours'] / t['Issues']
    return t.sort_values(['Project_Name', 'Hours'], ascending=[True, False])

type_hours  = hours_table(w, 'Type')
group_hours = hours_table(w, 'Type_Group')

print('--- by type ---');  print(type_hours.round(2).to_string(index=False))
print('--- by group ---'); print(group_hours.round(2).to_string(index=False))

Type_Group
Corrective (bugs)    10220
New functionality     5443
Tasks / technical     3514
Improvement           1677
Name: count, dtype: int64 

--- by type ---
            Project_Name                Type  Issues       Hours      Cost_USD  Share_Hours  Hours_per_Issue
     Appcelerator Studio                 Bug    1390  91,515.270 3,896,499.000        0.490           65.840
     Appcelerator Studio         Improvement     509  42,741.670 1,911,022.880        0.230           83.970
     Appcelerator Studio               Story     418  30,708.720 1,298,597.520        0.170           73.470
     Appcelerator Studio      Technical task     182  10,841.270   474,884.500        0.060           59.570
     Appcelerator Studio         New Feature      36   6,126.500   299,729.900        0.030          170.180
     Appcelerator Studio                Epic      10   2,677.730   118,063.800        0.010          267.770
     Appcelerator Studio            Sub-task      21   1,316.670    63,988

## 2.5 Tier utilization per year

Share of hours by tier, per project, by the year the work was resolved. Labels:
- **No tier (Other):** the contributor has an assignee but too few issues to be tiered (by design).
- **Unassigned:** the issue has no assignee; its cost uses the assumed rate.
- **Open:** issues with no resolution date, so they belong to no year.

What the mix shows:
- **Appcelerator** is Senior-led in every year (62–97%), with a junior bump in 2017–2018.
- **Mule:** in its three heaviest years (2016–2018) the Junior share grew from 2% to 11% to 24% while Senior fell from 54% to 28%.
- **Titanium** changes in phases: Junior/Junior-Mid in 2011–12, Mid in 2014–15, Mid-Senior in 2016–18 (55–79%), Senior again in 2019.

Cautions:
- Years with very few hours (for example Mule 2004–2006, Titanium 2020 with 735 h) are not reliable; filter them out in the dashboard.
- Each person has **one tier for the whole project**, so this shows who did the work by tier, not how people's skills changed over time.
- Seniority includes volume in its score, so heavy contributors tend to land in Senior. Part of the Senior dominance is built in by the definition.

In [ ]:
TIER_ORDER = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior', 'No tier (Other)', 'Unassigned']

w = prod[prod['Capped_Hours'] > 0].copy()          # all issues with effort, assigned or not

# year the work was finished; issues with no resolution date go in an 'Open' bucket
w['Year_Label'] = w['Resolution_Date'].dt.year.astype('Int64').astype(str).replace('<NA>', 'Open')

# tier label: real tier / assigned but not tiered / no assignee at all
w['Tier'] = w['Seniority_Tier'].astype(object)
w.loc[w['Tier'].isna() & w['Assignee_ID_Final'].notna(), 'Tier'] = 'No tier (Other)'
w['Tier'] = w['Tier'].fillna('Unassigned')

tier_year = (w.groupby(['Project_Name', 'Year_Label', 'Tier'])['Capped_Hours'].sum().reset_index())
tier_year['Share'] = tier_year['Capped_Hours'] / tier_year.groupby(['Project_Name', 'Year_Label'])['Capped_Hours'].transform('sum')

for p, d in tier_year.groupby('Project_Name'):
    print('\n===', p, '- share of hours by tier')
    t = d.pivot_table(index='Year_Label', columns='Tier', values='Share', fill_value=0)
    t = t.reindex(columns=[x for x in TIER_ORDER if x in t.columns]).round(2)
    t['Total_Hours'] = d.groupby('Year_Label')['Capped_Hours'].sum().round(0)
    print(t.to_string())
    print('Check: table total', round(d['Capped_Hours'].sum()), '| project total', round(w.loc[w['Project_Name'] == p, 'Capped_Hours'].sum()))


=== Appcelerator Studio - share of hours by tier
Tier        Junior  Junior-Mid   Mid  Senior  No tier (Other)  Total_Hours
Year_Label                                                                
2011         0.140       0.000 0.160   0.680            0.010    3,539.000
2012         0.080       0.000 0.240   0.630            0.050   22,102.000
2013         0.000       0.000 0.070   0.930            0.000   29,075.000
2014         0.100       0.000 0.000   0.900            0.000   24,787.000
2015         0.050       0.000 0.010   0.930            0.010   19,292.000
2016         0.040       0.000 0.000   0.960            0.000   15,085.000
2017         0.270       0.000 0.000   0.710            0.020   34,252.000
2018         0.180       0.030 0.000   0.620            0.170   21,563.000
2019         0.000       0.000 0.000   0.970            0.030    5,140.000
2020         0.000       0.000 0.000   0.780            0.220    2,990.000
Check: table total 177825 | project total 185928



## 2.6 Pareto: how concentrated is the effort?

People ranked by hours; how many are needed to reach 80% of the project's hours?

| Project | People | Needed for 80% | Top 5 share |
|---|---|---|---|
| Appcelerator Studio | 29 | 5 (17%) | 86% |
| Titanium Mobile Platform | 37 | 5 (14%) | 82% |
| Mule | 90 | 23 (26%) | 30% |

- Appcelerator and Titanium depend on a handful of people. The top contributor alone holds 38% (Appcelerator) and 27% (Titanium) of the hours: a key-person risk, taken further in the bus-factor analysis.
- Mule's effort is spread much more widely.
- Unassigned issues are excluded, since they belong to nobody. About 2% of each project's hours belong to contributors with no tier (the Other group); they are shown without a tier.

In [ ]:
w = prod[(prod['Capped_Hours'] > 0) & prod['Assignee_ID_Final'].notna()].copy()

par = (w.groupby(['Project_Name', 'Assignee_ID_Final'])
         .agg(Hours=('Capped_Hours', 'sum'),
              Issues=('ID', 'count'),
              Tier=('Seniority_Tier', 'first'))
         .reset_index()
         .sort_values(['Project_Name', 'Hours'], ascending=[True, False]))

par['Rank']     = par.groupby('Project_Name').cumcount() + 1
par['Share']    = par['Hours'] / par.groupby('Project_Name')['Hours'].transform('sum')
par['Cum_Pct']  = par.groupby('Project_Name')['Share'].cumsum()
par['People_Pct'] = par['Rank'] / par.groupby('Project_Name')['Rank'].transform('max')

# how concentrated is the effort?
summary = (par.groupby('Project_Name')
              .apply(lambda d: pd.Series({
                  'People': len(d),
                  'People_for_80pct_hours': int((d['Cum_Pct'] < 0.8).sum() + 1),
                  'Top5_share': d['Share'].head(5).sum()}),
                  include_groups=False))
summary['Pct_of_people_for_80pct'] = summary['People_for_80pct_hours'] / summary['People']
print(summary.round(2))

# who are the top 10 per project and what tier are they?
print(par.groupby('Project_Name').head(10)[['Project_Name', 'Rank', 'Assignee_ID_Final', 'Tier', 'Hours', 'Share', 'Cum_Pct']].round(2).to_string(index=False))

                          People  People_for_80pct_hours  Top5_share  \
Project_Name                                                           
Appcelerator Studio       30.000                   5.000       0.850   
Mule                      91.000                  23.000       0.300   
Titanium Mobile Platform  38.000                   5.000       0.820   

                          Pct_of_people_for_80pct  
Project_Name                                       
Appcelerator Studio                         0.170  
Mule                                        0.250  
Titanium Mobile Platform                    0.130  
            Project_Name  Rank  Assignee_ID_Final       Tier      Hours  Share  Cum_Pct
     Appcelerator Studio     1             165193     Senior 69,538.700  0.370    0.370
     Appcelerator Studio     2             165168     Senior 37,492.420  0.200    0.580
     Appcelerator Studio     3             165294     Senior 22,390.570  0.120    0.700
     Appcelerator Studio   

In [ ]:
no_tier = prod[prod['Assignee_ID_Final'].notna() & prod['Seniority_Tier'].isna()]
print(no_tier.groupby('Project_Name').agg(People=('Assignee_ID_Final', 'nunique'), Issues=('ID', 'count'), Hours=('Capped_Hours', 'sum')).round(0))

                          People  Issues      Hours
Project_Name                                       
Appcelerator Studio           22      87  6,533.000
Mule                          46    1775 14,535.000
Titanium Mobile Platform      25      97  6,242.000


### 2.7 Takeaways

- **Schedule:** SPI starts high in Appcelerator (3.44) and Titanium (2.81) and converges to 1.0, which is the effect of small early output over a short elapsed time. Mule is the only project that falls behind early (0.70 and 0.62) before recovering and peaking at 1.38.
- **Cost:** CPI ends at 0.43 (Appcelerator), 0.97 (Mule) and 0.61 (Titanium). Mule's cost per point is stable and close to the baseline (0.96–0.99 from 2017). Appcelerator and Titanium spent considerably more hours per point than their own history predicted.
- **Completeness:** % complete ends at 0.97, 1.02 and 1.09, so the earned work is close to the final scope in all three.
- **Scope growth:** 38–40 % of completed points were not in scope when their sprint started, mostly new bugs and work that had no estimate yet. The technical-debt proxy shows debt-related work is planned more often than it is added, so debt does not explain the growth.
- **Caveats:** The plan is a straight line and the final scope is known in hindsight. Hours are elapsed time capped at 720 h per issue, so costs are upper bounds. Appcelerator's EVM covers only 38 % of its pointed issues.

## 3. Seniority mix vs throughput and diminishing returns

**Question.** Does a team with more Seniors deliver more per person, and does adding Seniors stop paying off at some point?

**Definitions**
- **Throughput** = issues resolved in a sprint (per project, per sprint). Points come from `EV_Pts` in the EVM sprint table.
- **Heads** = distinct people who resolved at least one issue in the sprint (not the full roster, which TAWOS does not give per sprint).
- **Senior share** = Senior heads / heads. Issues without a tiered owner (about 4% of sprint issues) are left out.
- **Output per head** = issues (or points) / heads. This removes team size from the comparison.

**Choices**
- Only sprints with **3 or more heads** go into the fit. With 1-2 people the share can only be 0, 50% or 100%, which says nothing about a mix. That leaves 238 of 564 sprints.
- The curve is a **quadratic**. A negative squared term means diminishing returns; a positive one means the curve bends up. It is compared with a straight line (AIC) and its uncertainty comes from 1,000 bootstrap resamples of the sprints.

In [ ]:
# ---------- A1. Throughput per sprint and Senior share (feeds Tableau: A_throughput_sprint.csv) ----------
main = pd.read_csv('main_part3.csv')
MIN_HEADS = 3     # a share of 0 / 0.5 / 1 says little in a sprint with 1-2 people; fit only sprints with >= 3 people
RESOLVED  = ['Closed', 'Done', 'Resolved']

# Issues that belong to a sprint, were resolved, and have a tiered owner (~96% of them)
t = main[main['Sprint_ID'].notna() & main['Status'].isin(RESOLVED)
         & main['Resolution_Date'].notna() & main['Seniority_Tier'].notna()].copy()
t['Is_Senior'] = t['Seniority_Tier'].eq('Senior')

thr = (t.groupby(['Project_Name', 'Sprint_ID'])
         .agg(Issues_Resolved=('ID', 'count'), Heads=('Assignee_ID_Final', 'nunique'),
              Senior_Issues=('Is_Senior', 'sum'))
         .reset_index())
thr = thr.merge(t[t['Is_Senior']].groupby(['Project_Name', 'Sprint_ID'])['Assignee_ID_Final']
                  .nunique().rename('Senior_Heads').reset_index(), how='left')
thr['Senior_Heads'] = thr['Senior_Heads'].fillna(0)
thr['Senior_Share']       = thr['Senior_Heads'] / thr['Heads']          # share of PEOPLE who are Senior
thr['Senior_Issue_Share'] = thr['Senior_Issues'] / thr['Issues_Resolved']   # share of ISSUES done by Seniors
thr['Issues_per_Head']    = thr['Issues_Resolved'] / thr['Heads']

# Points come from the EVM sprint table (same EV_Pts used in the EVM section)
thr = thr.merge(evm[['Project_Name', 'Sprint_ID', 'Sprint_End', 'Year', 'EV_Pts']],
                on=['Project_Name', 'Sprint_ID'], how='left')
thr['Points_per_Head'] = thr['EV_Pts'] / thr['Heads']
thr['In_Fit'] = (thr['Heads'] >= MIN_HEADS) & thr['EV_Pts'].notna()

print("Sprints:", len(thr), "| with >= %d people and points (used in the fit):" % MIN_HEADS, int(thr['In_Fit'].sum()))
print(thr.groupby("Project_Name")[["Issues_Resolved", "Heads", "Senior_Share", "Issues_per_Head", "Points_per_Head"]].median().round(2).to_string())
print("\nSpearman correlation of Senior_Share with output per head (all sprints / sprints in the fit):")
for name, d in [('all', thr), ('in fit', thr[thr['In_Fit']])]:
    print(f"  {name:7s} issues/head {d['Senior_Share'].corr(d['Issues_per_Head'], method='spearman'):+.2f}"
          f" | points/head {d['Senior_Share'].corr(d['Points_per_Head'], method='spearman'):+.2f}")

Sprints: 564 | with >= 3 people and points (used in the fit): 238
                          Issues_Resolved  Heads  Senior_Share  Issues_per_Head  Points_per_Head
Project_Name                                                                                    
Appcelerator Studio                 6.000  2.000         1.000            3.500           14.670
Mule                               10.000  4.000         0.500            2.330            7.620
Titanium Mobile Platform            4.000  1.000         0.500            3.000           12.500

Spearman correlation of Senior_Share with output per head (all sprints / sprints in the fit):
  all     issues/head +0.15 | points/head +0.13
  in fit  issues/head +0.17 | points/head +0.05


### What the data shows

- **Senior share varies a lot between projects.** In a typical sprint with 3 or more people resolving issues, the average Senior share is **74% in Appcelerator, 46% in Mule and 41% in Titanium**. This is a snapshot of typical sprints, not a total over the life of the project.
- **Appcelerator is Senior-heavy but small.** Only 13 people resolve issues there, against 45 in Mule, so its share jumps more from sprint to sprint.
- **Most sprints are small.** The median sprint has 2-3 people resolving issues, so a single person changes the share a lot (with 3 people, each one is 33 points).

### How to read the Tableau chart

Each bar is **one sprint**. Its height is the **issues resolved** in that sprint, and its color is the **Senior share** in four steps: 0-25%, 26-50%, 51-75% and 76-100%.

- **Senior share means Senior vs everyone else.** It is the share of people who are Senior. Junior, Junior-Mid, Mid and Mid-Senior all count as "not Senior". Mid-Senior people are therefore not counted, which matters in Mule, where 31% of the people who resolved issues are Mid-Senior and 22% are Senior.
- **Heads** is the number of distinct people who resolved at least one issue in the sprint (1 to 16), not the whole team.
- **The chart is filtered to sprints with 3 or more heads.** Without the filter, 278 of 564 sprints (about half) are exactly 0% or 100% Senior, only because the team was 1 or 2 people. With the filter, 29 of 242 sprints (12%) are.
- **Reading the colors across a project.** A project with mostly dark bars (Appcelerator) staffed its sprints with a high share of Seniors. Mostly light bars mean the opposite. Bar height says how much was resolved, color says who resolved it. Taller dark bars do not by themselves prove that Seniors resolve more.

### Limits

- **Senior is partly built from volume.** The tier score includes issue volume, so the people with the most issues are Senior by construction. A Senior-heavy sprint is partly a sprint with prolific people.
- **Core team and one-off contributors are not separated.** One-off contributors own only 87 of the 20,854 issues, and 2 to 4 of them resolve issues per project, so they barely affect these results.
- **Small teams.** The median sprint has 2-3 heads, and only 32 sprints fall in the 76-100% band.
- **Sprint coverage.** Only 38% of Appcelerator's pointed issues belong to a sprint (81.5% Mule, 76% Titanium), so Appcelerator is the least represented.
- **Sprints without a date.** A few sprints are missing from the EVM sprint table, so they have no `Sprint_End` and do not appear on the time axis in Tableau.
- **A comparison is not a cause.** Nothing here shows that changing the mix changes output. Sprints with more Seniors may also have different work (bugs vs features).

In [ ]:
# ---------- A2. Diminishing returns: Senior share vs points per head, with a fitted curve ----------
fit = thr[thr['In_Fit']].copy()
x, y = fit['Senior_Share'].values, fit['Points_per_Head'].values

def poly_fit(x, y, deg):
    c = np.polyfit(x, y, deg)
    yhat = np.polyval(c, x)
    rss = ((y - yhat) ** 2).sum()
    n, k = len(y), deg + 1
    return c, rss, n * np.log(rss / n) + 2 * k                       # coefficients, RSS, AIC

c1, rss1, aic1 = poly_fit(x, y, 1)      # straight line = no diminishing returns
c2, rss2, aic2 = poly_fit(x, y, 2)      # curve: a negative squared term = diminishing returns
tot = ((y - y.mean()) ** 2).sum()
print(f"Linear     : slope {c1[0]:+.2f} pts/head per +100pp Senior | R2 {1 - rss1 / tot:.3f} | AIC {aic1:.1f}")
print(f"Quadratic  : a2 {c2[0]:+.2f} | R2 {1 - rss2 / tot:.3f} | AIC {aic2:.1f}")
peak = -c2[1] / (2 * c2[0]) if c2[0] != 0 else np.nan
shape = ('DIMINISHING RETURNS (curve bends down: concave)' if c2[0] < 0 else 'U-SHAPE (curve bends up: convex), NOT diminishing returns')
turn = f"turning point at Senior share = {peak:.2f} ({'maximum' if c2[0] < 0 else 'minimum'})" if 0 < peak < 1 else "no turning point inside 0-100% Senior"
print(f"Shape: {shape} | {turn}")

# Uncertainty: bootstrap the sprints 1,000 times and refit
rng = np.random.default_rng(42)
grid = np.linspace(0, 1, 21)
boots = np.array([np.polyval(np.polyfit(x[i], y[i], 2), grid)
                  for i in (rng.integers(0, len(x), len(x)) for _ in range(1000))])
curve = pd.DataFrame({'Senior_Share': grid, 'Fitted_Points_per_Head': np.polyval(c2, grid),
                      'CI_Low': np.percentile(boots, 5, axis=0), 'CI_High': np.percentile(boots, 95, axis=0)})
a2_boot = np.array([np.polyfit(x[i], y[i], 2)[0] for i in (rng.integers(0, len(x), len(x)) for _ in range(1000))])
print(f"Bootstrap 90% interval for the squared term: [{np.percentile(a2_boot, 5):+.1f}, {np.percentile(a2_boot, 95):+.1f}]"
      f"  (an interval that excludes 0 = evidence of curvature)")

# Binned medians: the same story without a model (this is what the Tableau chart shows next to the curve)
fit['Share_Band'] = pd.cut(fit['Senior_Share'], [-.01, .25, .5, .75, 1.0],
                           labels=['0-25%', '26-50%', '51-75%', '76-100%'])
bands = fit.groupby('Share_Band', observed=True).agg(Sprints=('Heads', 'size'), Median_Heads=('Heads', 'median'),
        Median_Issues_per_Head=('Issues_per_Head', 'median'), Median_Points_per_Head=('Points_per_Head', 'median'))
print("\n", bands.round(2).to_string())

# Second angle: output per head vs team size (the classic "add more people" curve)
sz = thr[thr['In_Fit']].assign(Team_Size=lambda d: d['Heads'].clip(upper=10))
print("\nMedian points per head by team size (10 = 10 or more):")
print(sz.groupby('Team_Size')['Points_per_Head'].agg(['size', 'median']).round(2).T)

thr.to_csv('A_throughput_sprint.csv', index=False)

Linear     : slope +6.33 pts/head per +100pp Senior | R2 0.018 | AIC 1185.9
Quadratic  : a2 +52.03 | R2 0.124 | AIC 1160.7
Shape: U-SHAPE (curve bends up: convex), NOT diminishing returns | turning point at Senior share = 0.45 (minimum)
Bootstrap 90% interval for the squared term: [+26.5, +78.3]  (an interval that excludes 0 = evidence of curvature)

             Sprints  Median_Heads  Median_Issues_per_Head  Median_Points_per_Head
Share_Band                                                                       
0-25%            59         5.000                   3.000                  11.500
26-50%           68         4.000                   2.200                   4.580
51-75%           79         5.000                   3.000                   9.330
76-100%          32         3.500                   3.500                  13.500

Median points per head by team size (10 = 10 or more):
Team_Size     3      4      5      6      7      8      9      10
size      75.000 45.000 42.000 3

In [ ]:
# ---------- A2b. Does the Senior share matter in each project? (same sprints as the fit: >= 3 people, with points) ----------
import warnings
BANDS = [-.01, .25, .50, .75, 1.0]
BAND_LABELS = ['0-25%', '26-50%', '51-75%', '76-100%']
rng = np.random.default_rng(42)

def boot(x, y, deg, n_boot=1000):
    """Bootstrap the sprints; return the 5th and 95th percentile of the highest-degree coefficient."""
    idx = (rng.integers(0, len(x), len(x)) for _ in range(n_boot))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                      # resamples with very few distinct shares are poorly conditioned
        coefs = [np.polyfit(x[i], y[i], deg)[0] for i in idx]
    return np.percentile(coefs, [5, 95])

rows, med = [], {}
for proj, d in thr[thr['In_Fit']].groupby('Project_Name'):
    x, y = d['Senior_Share'].values, d['Points_per_Head'].values
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        slope = np.polyfit(x, y, 1)[0]                       # points per head gained from 0% to 100% Senior
        a2 = np.polyfit(x, y, 2)[0]                          # squared term: + = U, - = arch
    lo, hi = boot(x, y, 1)
    qlo, qhi = boot(x, y, 2)
    rows.append({'Project_Name': proj, 'Sprints': len(d), 'Distinct_Shares': len(set(np.round(x, 2))),
                 'Slope': slope, 'Slope_Low': lo, 'Slope_High': hi,
                 'Spearman': pd.Series(x).corr(pd.Series(y), method='spearman'),
                 'R2_Line': np.corrcoef(x, y)[0, 1] ** 2,
                 'Squared_Term': a2, 'Squared_Low': qlo, 'Squared_High': qhi})
    med[proj] = (d.assign(Band=pd.cut(d['Senior_Share'], BANDS, labels=BAND_LABELS))
                   .groupby('Band', observed=False)['Points_per_Head'].agg(['size', 'median']))

res = pd.DataFrame(rows).set_index('Project_Name')
# Reading rule: the interval for the slope tells whether the trend is distinguishable from "no effect"
res['Slope_Reading'] = np.select([res['Slope_Low'] > 0, res['Slope_High'] < 0],
                                 ['Higher with more Seniors', 'Lower with more Seniors'], default='No clear trend')
res['Bend_Reading'] = np.select([res['Squared_Low'] > 0, res['Squared_High'] < 0],
                                ['U-shape (bends up)', 'Arch (diminishing returns)'], default='Bend not distinguishable from zero')
print("Change in points per head from 0% to 100% Senior (90% bootstrap interval) and rank correlation:")
print(res[['Sprints', 'Distinct_Shares', 'Slope', 'Slope_Low', 'Slope_High', 'Spearman', 'R2_Line', 'Slope_Reading']].round(2).to_string())
print("\nSquared term of the curve (90% bootstrap interval):")
print(res[['Squared_Term', 'Squared_Low', 'Squared_High', 'Bend_Reading']].round(1).to_string())
print("\nSprints and median points per head by Senior-share band:")
print(pd.concat(med, names=['Project_Name']).unstack('Band').round(1).to_string())

Change in points per head from 0% to 100% Senior (90% bootstrap interval) and rank correlation:
                          Sprints  Distinct_Shares  Slope  Slope_Low  Slope_High  Spearman  R2_Line             Slope_Reading
Project_Name                                                                                                                 
Appcelerator Studio            40                7 32.260     15.810      53.020     0.500    0.190  Higher with more Seniors
Mule                          186               25 -7.790    -14.250      -1.070    -0.190    0.030   Lower with more Seniors
Titanium Mobile Platform       12                4 -1.700    -11.090       4.790    -0.260    0.010            No clear trend

Squared term of the curve (90% bootstrap interval):
                          Squared_Term  Squared_Low  Squared_High                        Bend_Reading
Project_Name                                                                                         
Appcelerator Stud

### Does the Senior share matter in each project?

The combined curve mixes three very different projects, and **Mule is 186 of the 238 sprints (78%)**, so the combined shape is mostly Mule's. This cell repeats the analysis for each project separately, on the same sprints (3 or more people, with points).

**How to read the table**
- **Slope** = how many more (or fewer) points per head a sprint delivers when it goes from 0% to 100% Senior, with a straight line through that project's sprints.
- **90% interval** = the range where the slope probably lies, found by resampling the sprints 1,000 times. If the whole range is above zero, the trend is up; if it is all below zero, it is down; if it includes zero, the data cannot tell the trend from "no effect".
- **Squared term** = the bend of the curve. Positive means a U, negative means an arch (diminishing returns). The same interval rule applies.

| Project | Sprints | Slope (90% interval) | Rank correlation | What it shows |
|---|---|---|---|---|
| Appcelerator Studio | 40 | **+32** (+16 to +53) | +0.50 | Higher output per head with more Seniors |
| Mule | 186 | **-8** (-14 to -1) | -0.19 | Slightly **lower** output with more Seniors |
| Titanium Mobile Platform | 12 | -2 (-11 to +5) | -0.26 | No clear trend: the interval includes zero |

Median points per head by band of Senior share (number of sprints in brackets):

| Project | 0-25% | 26-50% | 51-75% | 76-100% |
|---|---|---|---|---|
| Appcelerator | 9.2 (1) | 9.9 (6) | 14.5 (19) | 24.7 (14) |
| Mule | 12.0 (55) | 4.0 (58) | 7.8 (55) | 3.1 (18) |
| Titanium | 1.8 (3) | 5.8 (4) | 0.0 (5) | none |

### What it means

- **Appcelerator:** the clearest result. Sprints with more Seniors delivered more points per person. This is the only project where the whole interval is above zero. But there are only 40 sprints, 1 sprint in the lowest band and 7 different share values, and the curve cannot be told apart from a straight line (squared-term interval -29 to +95). So "more Seniors went with more output" is safe, "more Seniors always helps" is not.
- **Mule:** the opposite direction, and weak (the line explains 3% of the variation). The sprints with the fewest Seniors have the highest median output (12.0) and the sprints with the most have the lowest (3.1). The curve is a U (squared term +28, interval +5 to +54), but the rise at the right edge rests on 18 sprints, so it should not be read as a benefit.
- **Titanium:** nothing can be said. 12 sprints and 4 different shares. The arch in the Tableau chart (which dips below zero, which is impossible) comes from so few points. The bend interval (-94 to +23) includes zero.
- **Together:** the data does not support "adding Seniors raises output per person". One project rises, one falls slightly, one is undecided, and the combined U is mostly Mule.

### Limits
- **Different projects, different work.** Appcelerator's sprints have a median of 2 people and Mule's of 4, and the work mix differs (bugs vs tasks), so the projects are not directly comparable.
- **Senior is partly built from issue volume**, so Senior-heavy sprints are partly sprints with prolific people.
- **Small samples.** With 40 and 12 sprints, a few sprints can move the result. The intervals show this: a wide interval means the answer is uncertain.
- **Several comparisons.** Looking at three projects and two measures makes it easier to find a pattern by chance. Treat each result as something to check, not as a finding.
- **Association, not cause.** Nothing here shows that adding a Senior to a team would change output.

## 4. Bottleneck: do issues wait on Senior review?

TAWOS has no field that says who reviewed an issue, so this is a **proxy**, built from the change log and the comments.

**What is measured**
- **Time in review** = hours from an issue entering a review-like status (*In Review*, *Code Review*, *In Code Review*, *To Be Reviewed*) until its next status change. "In  Review" with two spaces is the same status. Calendar hours, weekends included.
- **Owner tier** = the tier of the issue's final assignee. If Junior-owned issues wait longer than Senior-owned ones, that is consistent with a Senior bottleneck (but does not prove it).
- **Senior comments in review** = comments written by a Senior (other than the issue owner) while the issue was in review. Comment authors are matched to tiers through the team profile (about 78% of comments match).
- **Load test** = if Seniors were the bottleneck, months with more reviews per active Senior should show longer waits.

**Tableau.** `A_review_stints.csv` (one row per stint), `A_review_by_tier.csv` (summary) and `A_review_load_month.csv` (load vs wait).

In [ ]:
#  Bottleneck proxy: time in review-like statuses, by owner tier (feeds Tableau) ----------
REVIEW_STATES = {'in review', 'code review', 'in code review', 'to be reviewed'}   # lower-case, spaces collapsed
TIER_ORDER = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior']
MIN_N = 30                                                                       # fewer stints than this: shown but not interpreted

# Dates may arrive as text depending on how `main` was loaded: convert once so .dt works below
for c in ['Creation_Date', 'Resolution_Date', 'Sprint_Start', 'Sprint_End']:
    if c in main.columns:
        main[c] = pd.to_datetime(main[c], errors='coerce')

norm = lambda s: s.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()   # "In  Review" (2 spaces) = "In Review"

cl_a = pd.read_csv('Change_Log.csv', usecols=['Issue_ID', 'Field', 'To_String', 'Creation_Date'],
                   parse_dates=['Creation_Date'])
st = cl_a[cl_a['Field'] == 'status'].sort_values(['Issue_ID', 'Creation_Date']).copy()
st['To'] = norm(st['To_String'])
st['Left_At'] = st.groupby('Issue_ID')['Creation_Date'].shift(-1)                # the next status change ends the stint

rv = st[st['To'].isin(REVIEW_STATES)].rename(columns={'Creation_Date': 'Review_Start'}).copy()
print("Status changes:", len(st), "| entries into a review state:", len(rv), "| issues:", rv['Issue_ID'].nunique())
print("Stints with no later status change (left out):", int(rv['Left_At'].isna().sum()))
rv = rv.dropna(subset=['Left_At'])
rv['Review_Hours'] = (rv['Left_At'] - rv['Review_Start']).dt.total_seconds() / 3600     # calendar hours, weekends included

rv = rv.merge(main[['ID', 'Issue_Key', 'Project_Name', 'Seniority_Tier', 'Assignee_ID_Final']],
              left_on='Issue_ID', right_on='ID', how='left')
rv['Owner_Tier'] = rv['Seniority_Tier'].fillna('No tier (assumed rate)')
rv['Junior_Side'] = rv['Owner_Tier'].isin(['Junior', 'Junior-Mid'])

# Who commented while the issue was in review? (comments by someone other than the owner)
team_a = pd.read_csv('team_profiles_v1.csv', usecols=['Assignee_ID', 'Seniority_Tier'])
cm = pd.read_csv('Comment.csv', usecols=['Issue_ID', 'Author_ID', 'Creation_Date'], parse_dates=['Creation_Date'])
cm['Author_Tier'] = cm['Author_ID'].map(team_a.set_index('Assignee_ID')['Seniority_Tier'])
rv = rv.reset_index(drop=True)
j = rv[['Issue_ID', 'Review_Start', 'Left_At', 'Assignee_ID_Final']].reset_index(names='k').merge(cm, on='Issue_ID', suffixes=('', '_c'))
j = j[(j['Creation_Date'] >= j['Review_Start']) & (j['Creation_Date'] <= j['Left_At']) & (j['Author_ID'] != j['Assignee_ID_Final'])]
rv['Comments_by_Others']  = j.groupby('k').size().reindex(rv.index).fillna(0).astype(int)
rv['Senior_Comments']     = j[j['Author_Tier'] == 'Senior'].groupby('k').size().reindex(rv.index).fillna(0).astype(int)
rv['Has_Senior_Comment']  = rv['Senior_Comments'] > 0

q = lambda p: (lambda s: s.quantile(p))
by_tier = (rv.groupby('Owner_Tier')['Review_Hours']
             .agg(Stints='size', Median_Hours='median', P75_Hours=q(.75), P90_Hours=q(.90))
             .join(rv.groupby('Owner_Tier')['Has_Senior_Comment'].mean().rename('Pct_With_Senior_Comment'))
             .reindex(TIER_ORDER + ['No tier (assumed rate)']).dropna(how='all'))
by_tier['Enough_Data'] = by_tier['Stints'] >= MIN_N
print("\nTime in review by owner tier (hours):")
print(by_tier.round(2).to_string())

by_proj = (rv.groupby(['Project_Name', 'Junior_Side'])['Review_Hours']
             .agg(Stints='size', Median_Hours='median', P90_Hours=q(.90)).round(1))
print("\nBy project, Junior side (Junior + Junior-Mid) vs the rest:")
print(by_proj.to_string())

# A real bottleneck grows with load: more reviews per active Senior -> longer waits
sen = main[(main['Seniority_Tier'] == 'Senior') & main['Resolution_Date'].notna()].copy()
sen['Month'] = sen['Resolution_Date'].dt.to_period('M')
heads = sen.groupby(['Project_Name', 'Month'])['Assignee_ID_Final'].nunique().rename('Active_Senior_Heads')
rv['Month'] = rv['Review_Start'].dt.to_period('M')
load = (rv.groupby(['Project_Name', 'Month']).agg(Reviews_Started=('Review_Hours', 'size'), Median_Review_Hours=('Review_Hours', 'median'))
          .join(heads).dropna().reset_index())
load = load[load['Active_Senior_Heads'] > 0].copy()
load['Reviews_per_Senior'] = load['Reviews_Started'] / load['Active_Senior_Heads']
load['Month'] = load['Month'].dt.to_timestamp()

print("\nSpearman: reviews per active Senior vs median review hours, by project-month")
for p, d in load.groupby('Project_Name'):
    print(f"  {p:26s} months {len(d):3d}  rho {d['Reviews_per_Senior'].corr(d['Median_Review_Hours'], method='spearman'):+.2f}")
print(f"  {'All':26s} months {len(load):3d}  rho {load['Reviews_per_Senior'].corr(load['Median_Review_Hours'], method='spearman'):+.2f}")

# Same question in three equal groups of months (Spearman cannot see a bump, the table can)
load['Load_Band'] = pd.qcut(load['Reviews_per_Senior'].rank(method='first'), 3, labels=['Low load', 'Mid load', 'High load'])
print("\nMedian review hours by load group (equal-sized groups of project-months):")
print(load.groupby('Load_Band', observed=True)
          .agg(Project_Months=('Month', 'size'), Load_Min=('Reviews_per_Senior', 'min'),
               Load_Max=('Reviews_per_Senior', 'max'), Median_Review_Hours=('Median_Review_Hours', 'median')).round(1).to_string())

# Only the stints file goes to Tableau: tier summaries are calculated there; the load check stays in Python
rv.drop(columns=['ID', 'Month']).to_csv('A_review_stints.csv', index=False)

Status changes: 53107 | entries into a review state: 1031 | issues: 982
Stints with no later status change (left out): 7

Time in review by owner tier (hours):
                        Stints  Median_Hours  P75_Hours  P90_Hours  Pct_With_Senior_Comment  Enough_Data
Owner_Tier                                                                                              
Junior                     122        25.580    142.270  1,169.240                    0.200         True
Junior-Mid                  16        55.970    193.920    289.520                    0.060        False
Mid                        190        50.490    300.930  4,949.610                    0.030         True
Mid-Senior                  15        96.820    203.170    946.400                    0.000        False
Senior                     642        25.130    124.550    332.630                    0.200         True
No tier (assumed rate)      39        91.780    366.900    848.330                    0.280         True


## Is there a bottleneck waiting on Senior review?

### What we can and cannot measure

The data does not say who reviewed an issue. So this is a **proxy**: how long an issue sat in a review-like status (`In Review`, `Code Review`, `In Code Review`, `To Be Reviewed`) before its next status change, taken from the change log. If Seniors were a bottleneck, issues would wait longer in review and the wait would grow when Seniors are busier.

- **Stint** = one visit of an issue to a review status. The next status change ends it. Hours are calendar hours (weekends included).
- 982 issues entered a review state: Appcelerator 549 stints, Titanium 441, Mule 34. Seven stints with no later status change are left out.
- **Owner tier** = the tier of the person who owns the issue, not of the reviewer.

### How `Has_Senior_Comment` is calculated

- It is **True** when, during one review stint, someone **other than the issue's owner** left a comment, and that commenter maps to **Senior** in `team_profiles_v1.csv`.
- Authors who are not in that file (clients, reporters, outside users) get no tier, so they are **never counted as Senior**. Client comments are not mixed in.
- It is a hint that a Senior looked at the issue, not proof that a Senior did the review. A Senior commenting on another Senior's issue counts too (peer review).
- The Senior tier is partly built from issue volume, so some "Seniors" may simply be prolific contributors.

### What the data shows

- **Review wait does not rise with the owner's tier.** Median hours in review are 25.6 for Junior-owned issues and 25.1 for Senior-owned ones. Mid-owned issues are slower (median 50.5), but with a very long tail (P90 of about 4,950 hours), so a few stuck issues drive it.
- **Senior comments in review are an Appcelerator pattern, not a general one.** Across all stints, a Senior commented in 20% of Junior-owned and 20% of Senior-owned stints, and in 3% of Mid-owned ones. But by project:

  | Project | Stints | Senior commented |
  |---|---|---|
  | Appcelerator | 549 | 30% |
  | Mule | 34 | 3% |
  | Titanium | 441 | 1% |

  Inside Appcelerator, Junior-owned stints (77) show 32% and Senior-owned stints (453) show 28%, so there is no sign that Juniors get more Senior attention. The Mid figure (3%) comes almost entirely from Titanium, where Senior comments are rare for every tier. In Mule and Titanium there are too few Senior comments to test the question.
- **Why Appcelerator stands out.** It is the most Senior-heavy project (74% average Senior share), so more Seniors are around to comment, and its team may simply discuss work inside Jira more. This is a difference between projects, not between tiers.
- **Tiers with fewer than 30 stints are shown but not interpreted** (for example Junior-Mid, 16 stints).
- **Mule has only 34 stints**, so the tier medians are mostly Appcelerator and Titanium.

### Does more review load make reviews slower? (calculated in Python, not plotted)

**Load** = reviews started in a month ÷ Seniors active that month, per project. "Active Senior" means a Senior who resolved at least one issue that month. **Wait** = the median hours the month's reviews spent in a review state.

Across 128 project-months (Appcelerator 65, Titanium 50, Mule 13), load goes from 0.1 to 36 reviews per Senior, with a median of 3. Splitting the months into three equal groups by load:

| Load group | Project-months | Reviews per Senior | Median review hours |
|---|---|---|---|
| Low | 43 | 0.1 to 1.5 | 46.1 |
| Mid | 42 | 1.5 to 5.0 | 83.6 |
| High | 43 | 5.0 to 36 | 27.7 |

**Reading it.** If Seniors were a bottleneck, the high-load months would have the longest waits. They have the shortest. The rank correlation (Spearman) is **−0.03**, which means no steady relationship. The middle group is the slowest, but that is a bump, not a trend that grows with load. Spearman cannot see a bump like this, so the table is the better check.

**Why this is not a Tableau chart.** The load needs the count of active Seniors per month, which comes from `projects_dataset`. The waits come from the change log. Joining the two in Tableau adds complexity for a result that shows no pattern, so the calculation stays in Python as a check.

### Verdict

**No sign of a Senior-review bottleneck in this data.** Waits do not grow with the owner's tier, Senior comments do not favor Junior work where they can be tested (Appcelerator), and busier months are not slower. This is evidence of absence only within what the proxy can see, not proof that no bottleneck exists.

### How to read the Tableau charts

Only `A_review_stints.csv` goes to Tableau. Each row is one stint. The tier summaries (median hours, share with a Senior comment) are calculated there from the rows, so no separate summary file is needed.

- **Do not stack the bars when Project is on Color.** A stacked bar adds each project's own rate, so the total height is not a real rate (it can look like 72% when no project rate is that high). Turn off **Análisis → Apilar marcas**, or put `Project_Name` on Columns so each project has its own bar.
- **Check the number of stints behind each bar.** Appcelerator and Titanium dominate the counts and Mule has 34.

### Limits

- **Reviewer unknown.** A status change does not say who reviewed. Owner tier is used as a stand-in for who the work belongs to.
- **Senior comments depend on the project.** Most Senior comments in review are in Appcelerator, so that result cannot be generalized to Mule or Titanium.
- **Thin and uneven.** Many months have only 1 to 3 reviews, so the median wait is noisy. Mule contributes 34 stints and 13 project-months.
- **Calendar hours.** A stint over a weekend or holiday looks longer than the working time it took.
- **Long tails.** Some issues sit in a review status for months (P90 above 1,000 hours for Junior, near 4,950 for Mid). Those are often forgotten tickets, not real waiting, so medians are used instead of averages.
- **Shared load.** "Per Senior" assumes the active Seniors share the reviews evenly.
- **Not a cause.** A flat result here does not show that adding or removing Seniors would change review speed.

## 5. Bug density and rework

In [ ]:
# ----------  Bug density by project and creation year (Tableau computes this from projects_dataset; no CSV needed) ----------
MIN_ISSUES = 30                                   # project-years with fewer issues are shown but not interpreted

main['Creation_Date'] = pd.to_datetime(main['Creation_Date'], errors='coerce')
bd = main[main['Creation_Date'].notna()].copy()
bd['Year']   = bd['Creation_Date'].dt.year
bd['Is_Bug'] = bd['Type'].eq('Bug')

# Overall, per project
by_proj = bd.groupby('Project_Name')['Is_Bug'].agg(Issues='size', Bugs='sum')
by_proj['Bug_Density'] = by_proj['Bugs'] / by_proj['Issues']
print("Bug density by project (bugs / all issues):")
print(by_proj.round(2).to_string())

# Per project and creation year
by_year = bd.groupby(['Project_Name', 'Year'])['Is_Bug'].agg(Issues='size', Bugs='sum')
by_year['Bug_Density']   = by_year['Bugs'] / by_year['Issues']
by_year['Enough_Issues'] = by_year['Issues'] >= MIN_ISSUES
print("\nBug density by project and creation year:")
print(by_year.round(2).to_string())

Bug density by project (bugs / all issues):
                          Issues  Bugs  Bug_Density
Project_Name                                       
Appcelerator Studio         5979  3455        0.580
Mule                       11816  5421        0.460
Titanium Mobile Platform    3059  1344        0.440

Bug density by project and creation year:
                               Issues  Bugs  Bug_Density  Enough_Issues
Project_Name             Year                                          
Appcelerator Studio      2011     886   550        0.620           True
                         2012    1546   795        0.510           True
                         2013    1466   731        0.500           True
                         2014     780   458        0.590           True
                         2015     615   412        0.670           True
                         2016     234   172        0.740           True
                         2017     242   179        0.740           True
     

In [ ]:
# ---------- 5.2 Rework: reopen events by project x owner tier, bugs vs everything else (feeds Tableau) ----------
MIN_RESOLVED = 30                                  # groups with fewer resolved issues are shown but not interpreted
TIER_ORDER = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior', 'No tier']

norm = lambda s: s.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
cl_b = pd.read_csv('Change_Log.csv', usecols=['Issue_ID', 'Field', 'To_String'])
st_b = cl_b[cl_b['Field'] == 'status']
ro = st_b[norm(st_b['To_String']) == 'reopened']
reopens = ro.groupby('Issue_ID').size().rename('Reopen_Count')          # EVERY reopen event counts: 3 reopens = 3
print("Reopen events:", int(reopens.sum()), "| issues reopened at least once:", len(reopens))

rw = main[main['Resolution_Date'].notna()].copy()                         # only issues that were resolved
rw = rw.merge(reopens, left_on='ID', right_index=True, how='left')
rw['Reopen_Count'] = rw['Reopen_Count'].fillna(0).astype(int)
rw['Was_Reopened'] = rw['Reopen_Count'] > 0
rw['Owner_Tier']   = rw['Seniority_Tier'].fillna('No tier')
rw['Issue_Kind']   = np.where(rw['Type'].eq('Bug'), 'Bug', 'Not a bug')

rework = (rw.groupby(['Project_Name', 'Issue_Kind', 'Owner_Tier'])
            .agg(Resolved=('ID', 'count'), Reopen_Events=('Reopen_Count', 'sum'), Issues_Reopened=('Was_Reopened', 'sum'))
            .reset_index())
rework['Events_per_100']  = 100 * rework['Reopen_Events'] / rework['Resolved']    # can exceed the % of issues reopened
rework['Pct_Reopened']    = rework['Issues_Reopened'] / rework['Resolved']
rework['Enough_Resolved'] = rework['Resolved'] >= MIN_RESOLVED
rework['Owner_Tier'] = pd.Categorical(rework['Owner_Tier'], TIER_ORDER, ordered=True)
rework = rework.sort_values(['Project_Name', 'Issue_Kind', 'Owner_Tier'])

print("\nRework by project, issue kind and owner tier:")
print(rework.round(2).to_string(index=False))

# Only the per-issue reopen counts go to Tableau: relate on ID to projects_dataset, which already has project, type and tier
reopens.reset_index().rename(columns={'Issue_ID': 'ID'}).to_csv('B_reopen_counts.csv', index=False)

Reopen events: 3369 | issues reopened at least once: 2627

Rework by project, issue kind and owner tier:
            Project_Name Issue_Kind Owner_Tier  Resolved  Reopen_Events  Issues_Reopened  Events_per_100  Pct_Reopened  Enough_Resolved
     Appcelerator Studio        Bug     Junior       285             73               56          25.610         0.200             True
     Appcelerator Studio        Bug Junior-Mid        15              0                0           0.000         0.000            False
     Appcelerator Studio        Bug        Mid       615            244              179          39.670         0.290             True
     Appcelerator Studio        Bug     Senior      2176            928              623          42.650         0.290             True
     Appcelerator Studio        Bug    No tier        62             24               19          38.710         0.310             True
     Appcelerator Studio  Not a bug     Junior       117             20        

### What we measure

- **Bug** = an issue whose `Type` is `Bug`. All issues count, not only those in sprints: sprint tagging only starts in 2013 (Appcelerator, Mule) and 2014 (Titanium), so a sprint-only view would drop most of the history.
- **Bug density** = bugs ÷ all issues, by project and creation year. It is the share of the work log that is bug work, not a measure of code quality.
- **Rework** = how often resolved issues were reopened. A **reopen event** is every move into a `Reopened` status in the change log. An issue reopened three times counts as three events.
- **Events per 100** = reopen events ÷ resolved issues × 100. It can be higher than the share of issues reopened, because one issue can come back more than once.
- **Share of issues reopened** = issues reopened at least once ÷ resolved issues.
- Only resolved issues are in the denominator. The rework figures are split into **bugs** and **not a bug** (features, tasks, improvements), by owner tier and project.

### What the data shows

**Bug density by project**

| Project | Issues | Bugs | Bug density |
|---|---|---|---|
| Appcelerator | 5,979 | 3,455 | 58% |
| Mule | 11,816 | 5,421 | 46% |
| Titanium | 3,059 | 1,344 | 44% |

By creation year, the trend differs by project: Appcelerator rises from about 50% (2012–2013) to 74–76% (2016–2018); Mule sits at 45–65% until 2012, then falls to 32% in 2016 and 39% in 2017; Titanium falls from 87% in 2011 to 17% in 2017. Project-years with fewer than 30 issues (Mule 2019–2020, and 2020 in the other two) are shown but not interpreted.

**Rework by project.** There are 3,369 reopen events across 2,627 issues.

| Project | Resolved issues | Reopened at least once | Events per 100 resolved |
|---|---|---|---|
| Appcelerator | 5,152 | 25.5% | about 36 |
| Mule | 11,292 | 9.8% | about 12 |
| Titanium | 2,774 | 5.7% | about 7 |

A reopened bug in Appcelerator comes back about 1.4 times on average (43 events per 100 Senior bugs, but 29% of them reopened).

**Rework by owner tier** (events per 100 resolved, with the share reopened in brackets):

| Project | Kind | Junior | Mid | Mid-Senior | Senior |
|---|---|---|---|---|---|
| Appcelerator | Bug | 26 (20%) | 40 (29%) | none | 43 (29%) |
| Appcelerator | Not a bug | 17 (13%) | 34 (25%) | none | 28 (22%) |
| Mule | Bug | 8 (7%) | 19 (15%) | 14 (12%) | 13 (12%) |
| Mule | Not a bug | 18 (12%) | 11 (9%) | 11 (10%) | 11 (10%) |
| Titanium | Bug | 10 (9%) | 14 (9%) | 6 (6%) | 8 (7%) |
| Titanium | Not a bug | 0 (0%) | 9 (7%) | 2 (2%) | 6 (6%) |

- **The project explains more than the tier.** Appcelerator reopens about three times as much as Mule and five times as much as Titanium, in every tier. Compare tiers only within a project.
- **No tier is consistently best or worst.** In Mule, Junior bugs have the lowest rework (8) but Junior non-bugs the highest (18). The order flips with the kind of work, so there is no "more senior means less rework" pattern.
- **Appcelerator Junior-Mid (15 resolved bugs) is too small to read.** It is flagged in the Python table.

### What the "NULL" tier is

The NULL bars on the chart are the 1,679 resolved issues with no seniority tier. They are not one group:

| Reason | Appcelerator | Mule | Titanium |
|---|---|---|---|
| Other contributor (one-off, not tiered) | 77 | 176 | 79 |
| Unassigned and not worked (closed without a fix or still open) | 0 | 1,214 | 0 |
| Unassigned, worked, owner unknown | 4 | 127 | 2 |

In Appcelerator and Titanium, NULL is almost entirely one-off contributors (about 80 issues each). In Mule, most of it is unassigned issues that were never worked, which is why Mule's NULL bars are low: there was little work to reopen. NULL says nothing about seniority and should not be compared with the tiers.

### How to read the Tableau chart

Each project has a panel, each tier two side-by-side bars (bugs and not a bug), and the height is reopen events per 100 resolved issues. Only `B_reopen_counts.csv` (issue ID and reopen count) comes from outside `projects_dataset`; the project, type and tier come from `projects_dataset`.
- The sheet needs a filter on `Resolution_Date` excluding Null, or unresolved issues will be counted too.
- Bars are not stacked: stacked bars would add the two kinds together.
- The NULL tier should be excluded or labeled clearly.

### Verdict

**Rework changes far more by project than by seniority tier.** Within a project, no tier is consistently best or worst, so these data do not show that higher tiers produce more durable work. Bug density rises in Appcelerator and falls in Mule and Titanium over time, but that is a change in what the log contains, not proof that quality got better or worse.

### Limits

- **Owner is not author.** The tier is the person who owned the issue. For a reopened bug, it is who fixed it; for a reopened feature, who built it. The data cannot say who introduced a bug.
- **A reopen has many causes.** It can be a failed fix, a regression, new information, or a workflow habit. Appcelerator's much higher rate may reflect how the team used the `Reopened` status.
- **Bug density depends on the denominator.** It falls when the log adds more features or tasks, even if the number of bugs does not change.
- **The Senior tier is partly built from volume,** so a "Senior" can simply be a prolific contributor.
- **Thin groups.** Appcelerator has no Mid-Senior people, and a few tier groups (Junior-Mid in Appcelerator, Junior in Titanium non-bugs) are small.
- **Reopened and still open.** Issues reopened and not yet resolved again are left out of the rework rate.
- **Not a cause.** Nothing here shows that changing the staffing mix would change rework.

## 6. Planned vs actual effort by owner tier

In [ ]:
# ---------- 6.1 Planned vs actual effort by owner tier  ----------
MIN_ISSUES = 30                                   # groups with fewer issues are shown but not interpreted
TIER_ORDER = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior', 'No tier']

src = prod.copy()                                 # prod = projects_dataset.csv, which still has Planned_Hours
src['Resolution_Date'] = pd.to_datetime(src['Resolution_Date'], errors='coerce')

# Resolved, pointed issues WITH logged effort. Issues with 0 hours would look "under plan" only because nobody logged time.
pa = src[src['Resolution_Date'].notna() & src['Planned_Hours'].notna() & (src['Capped_Hours'] > 0)].copy()
pa['Owner_Tier'] = pa['Seniority_Tier'].fillna('No tier')
pa['Ratio']      = pa['Capped_Hours'] / pa['Planned_Hours']          # >1 = took more hours than planned for its points
pa['Over_Plan']  = pa['Ratio'] > 1

n_all = int((src['Resolution_Date'].notna() & src['Planned_Hours'].notna()).sum())
print(f"Resolved issues with a plan: {n_all} | with logged effort (used here): {len(pa)}")

pva = (pa.groupby(['Project_Name', 'Owner_Tier'])
         .agg(Issues=('ID', 'count'), Planned_Hours=('Planned_Hours', 'sum'), Actual_Hours=('Capped_Hours', 'sum'),
              Median_Ratio=('Ratio', 'median'), Pct_Over_Plan=('Over_Plan', 'mean'))
         .reset_index())
pva['Actual_vs_Planned'] = pva['Actual_Hours'] / pva['Planned_Hours']      # 1.0 = on plan, 1.2 = 20 % more hours than planned
pva['Enough_Issues']     = pva['Issues'] >= MIN_ISSUES
pva['Owner_Tier'] = pd.Categorical(pva['Owner_Tier'], TIER_ORDER, ordered=True)
pva = pva.sort_values(['Project_Name', 'Owner_Tier'])

print("\nPlanned vs actual hours by project and owner tier:")
print(pva.round(2).to_string(index=False))

Resolved issues with a plan: 8475 | with logged effort (used here): 5798

Planned vs actual hours by project and owner tier:
            Project_Name Owner_Tier  Issues  Planned_Hours  Actual_Hours  Median_Ratio  Pct_Over_Plan  Actual_vs_Planned  Enough_Issues
     Appcelerator Studio     Junior     129      8,921.210    18,258.100         1.120          0.540              2.050           True
     Appcelerator Studio Junior-Mid       1         86.630       720.000         8.310          1.000              8.310          False
     Appcelerator Studio        Mid     313      8,946.580     7,793.250         0.110          0.220              0.870           True
     Appcelerator Studio     Senior    1882     87,830.750   135,168.580         0.390          0.350              1.540           True
     Appcelerator Studio    No tier      17      1,560.660     5,952.670         1.960          0.710              3.810          False
                    Mule     Junior      79      4,460.640 

### What we measure

- **Planned hours** (`Planned_Hours`) = the issue's story points × the project's rolling 12-month hours per point, taken at the issue's estimation month. It is a size-based expectation built from past actual hours, **not a plan someone wrote before the work**.
- **Actual hours** = capped logged hours of the issue (capped at 720).
- **Actual vs planned** = total actual hours ÷ total planned hours for a tier in a project. 1.0 means on the expected hours for the size of the work, 2.0 means twice as many hours.

### How much of the data this covers

Planned hours exist only for issues that have **story points and an estimation date**: 8,873 of 20,854 issues (42.5%). The chart uses a smaller group still:

| Step | Issues |
|---|---|
| All issues | 20,854 |
| With story points and an estimation date | 8,873 |
| ...resolved | 8,475 |
| ...with logged effort (**used in the chart**) | 5,798 (27.8% of all issues) |

Less than a third of the issues are therefore behind this chart. The other 72% either have no points (so no planned hours) or no logged hours. The EVM has the same limit, because it also needs story points to earn value. Issues with 0 logged hours are left out on purpose: they would look "under plan" only because nobody logged time.

### What the data shows

Actual ÷ planned hours by owner tier (groups under 30 issues in brackets):

| Project | Junior | Junior-Mid | Mid | Mid-Senior | Senior |
|---|---|---|---|---|---|
| Appcelerator | 2.05 | (1 issue) | 0.87 | none | 1.54 |
| Mule | 3.02 | 2.43 | 2.25 | 1.39 | 1.14 |
| Titanium | (23) 2.59 | (3) | 1.61 | 1.37 | 1.11 |

- **Mule and Titanium step down with seniority**: Juniors log about 2.5 to 3 times the expected hours for their size, Seniors about 1.1.
- **Appcelerator does not follow that pattern** (Mid 0.87, Senior 1.54).
- **A few long issues drive the totals.** The median per-issue ratio is well below the total in most groups (for example Appcelerator Senior: 0.39 vs 1.54), so the typical issue came in under the expected hours while a few ran far over.
- The totals sit above 1 for every project (1.56, 1.36, 1.33) partly because issues with no logged hours were removed.

### How to read the Tableau chart

Each panel is a project and each bar a tier. Compare **bars within a project**, not their distance from 1.0: the plan is the project's own average, so the project as a whole is near 1 by construction. The tooltip shows the number of issues behind each bar. NULL (no tier) should be excluded or labeled, as in B.

### Limits

- **The plan comes from past actual hours,** so the comparison is partly circular: only differences between tiers carry information.
- **It uses today's story points,** not the original estimate, so re-estimated issues are compared with a number that was not known when work began.
- **Coverage is under a third** of the issues (see above), and it favors issues that have points and logged time.
- **Tiers differ in the work they get and in how much time they log.** A higher ratio can mean slower work, a smaller estimate for the same work, or different logging habits.
- **The Senior tier is partly built from issue volume.**
- **Thin groups:** Appcelerator Junior-Mid (1), Titanium Junior (23) and Junior-Mid (3).
- **Not a cause.** Nothing here shows that changing the mix would change hours.

In [ ]:
# ---------- 6.2 Late vs sprint end: delivered issues resolved after the end of their sprint (Tableau computes this from projects_dataset) ----------
MIN_ISSUES = 30                                   # groups with fewer issues are shown but not interpreted
GRACE_DAYS = 14                                   # "clearly late" = resolved more than 14 days after the sprint ended
DELIVERED  = ['Fixed', 'Done', 'Completed']       # work that was actually delivered (closed-without-fix issues have no delivery date to miss)
TIER_ORDER = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior', 'No tier']

lt = prod.copy()                                  # prod = projects_dataset.csv
for c in ['Resolution_Date', 'Sprint_Start', 'Sprint_End']:
    lt[c] = pd.to_datetime(lt[c], errors='coerce')

n_sprint = int(lt['Sprint_ID'].notna().sum())
lt = lt[lt['Sprint_ID'].notna() & lt['Sprint_End'].notna() & lt['Resolution_Date'].notna()]
n_resolved = len(lt)
lt = lt[lt['Resolution'].isin(DELIVERED)]
n_delivered = len(lt)
lt = lt[lt['Resolution_Date'] >= lt['Sprint_Start']].copy()      # resolved before the sprint began = not worked in that sprint
print(f"Issues in a sprint: {n_sprint} | resolved: {n_resolved} | delivered: {n_delivered} | used (resolved on/after sprint start): {len(lt)}")

lt['Days_After_End'] = (lt['Resolution_Date'] - lt['Sprint_End']).dt.total_seconds() / 86400   # negative = finished before the sprint ended
lt['Late']       = lt['Days_After_End'] > 0
lt['Late_Grace'] = lt['Days_After_End'] > GRACE_DAYS
lt['Owner_Tier'] = lt['Seniority_Tier'].fillna('No tier')
lt['Year']       = lt['Sprint_End'].dt.year

p90 = lambda s: s.quantile(.90)
def summarize(keys):
    t = (lt.groupby(keys)
           .agg(Issues=('ID', 'count'), Pct_Late=('Late', 'mean'), Pct_Late_Over_Grace=('Late_Grace', 'mean'),
                Median_Days=('Days_After_End', 'median'), P90_Days=('Days_After_End', p90))
           .reset_index())
    t['Enough_Issues'] = t['Issues'] >= MIN_ISSUES
    return t

print("\nBy project:")
print(summarize(['Project_Name']).round(2).to_string(index=False))

by_tier = summarize(['Project_Name', 'Owner_Tier'])
by_tier['Owner_Tier'] = pd.Categorical(by_tier['Owner_Tier'], TIER_ORDER, ordered=True)
print("\nBy project and owner tier:")
print(by_tier.sort_values(['Project_Name', 'Owner_Tier']).round(2).to_string(index=False))

print("\nBy project and sprint-end year:")
print(summarize(['Project_Name', 'Year']).round(2).to_string(index=False))

Issues in a sprint: 6181 | resolved: 5807 | delivered: 5253 | used (resolved on/after sprint start): 5077

By project:
            Project_Name  Issues  Pct_Late  Pct_Late_Over_Grace  Median_Days  P90_Days  Enough_Issues
     Appcelerator Studio    1101     0.200                0.100       -4.540    17.280           True
                    Mule    3095     0.340                0.210       -2.230   267.940           True
Titanium Mobile Platform     881     0.270                0.150       -2.800    33.050           True

By project and owner tier:
            Project_Name Owner_Tier  Issues  Pct_Late  Pct_Late_Over_Grace  Median_Days  P90_Days  Enough_Issues
     Appcelerator Studio     Junior      83     0.270                0.080       -4.040     9.250           True
     Appcelerator Studio Junior-Mid       1     0.000                0.000       -3.040    -3.040          False
     Appcelerator Studio        Mid      23     0.130                0.130       -7.170    49.580         

In [ ]:
## Why are issues late? Year-adjusted tier comparison + three "not the cause" tests
from scipy.stats import spearmanr

# Same filters as the C2 cell. Rebuilt here so the cell runs on its own.
DELIVERED = ['Fixed', 'Done', 'Completed']
lt = prod.copy()
for c in ['Resolution_Date', 'Sprint_Start', 'Sprint_End']:
    lt[c] = pd.to_datetime(lt[c], errors='coerce')
lt = lt[lt['Sprint_ID'].notna() & lt['Sprint_End'].notna() & lt['Resolution_Date'].notna()]
lt = lt[lt['Resolution'].isin(DELIVERED) & (lt['Resolution_Date'] >= lt['Sprint_Start'])].copy()
lt['Late'] = (lt['Resolution_Date'] - lt['Sprint_End']).dt.total_seconds() > 0
lt['Owner_Tier'] = lt['Seniority_Tier'].fillna('No tier')
lt['Year'] = lt['Sprint_End'].dt.year
lt['Is_Bug'] = lt['Type'].astype(str).str.contains('Bug', case=False)
lt['Sprint_Days'] = (lt['Sprint_End'] - lt['Sprint_Start']).dt.days
print(f"Issues used: {len(lt)} of {len(prod)} ({len(lt)/len(prod):.0%})")

# 1) Year-adjusted tier comparison: expected late rate = the project's late rate in that year
lt['Expected_Late'] = lt.groupby(['Project_Name', 'Year'])['Late'].transform('mean')
adj = (lt.groupby(['Project_Name', 'Owner_Tier'])
         .agg(Issues=('ID', 'count'), Actual_Late=('Late', 'mean'), Expected_Late=('Expected_Late', 'mean'))
         .reset_index())
adj['Gap_pts'] = (adj['Actual_Late'] - adj['Expected_Late']) * 100
adj['Enough_Issues'] = adj['Issues'] >= 30
print("\n1) Tier vs what its years predict (gap > 0 = later than the year explains)")
print(adj.round(2).to_string(index=False))

# 2) and 3) By project and year: bug share and sprint length next to the late rate
yr = (lt.groupby(['Project_Name', 'Year'])
        .agg(Issues=('ID', 'count'), Late=('Late', 'mean'), Bug_Share=('Is_Bug', 'mean'),
             Sprint_Days=('Sprint_Days', 'median')).reset_index())
yr = yr[yr['Issues'] >= 30]
print("\n2) and 3) By project and year (years with 30+ issues)")
print(yr.round(2).to_string(index=False))

# 4) Per-sprint: issues in the sprint vs late rate (sprints with 5+ issues)
sp = (lt.groupby(['Project_Name', 'Sprint_ID'])
        .agg(Issues=('ID', 'count'), Late=('Late', 'mean')).reset_index())
sp = sp[sp['Issues'] >= 5]
print("\nSpearman correlations with late rate (close to 0 = no relationship)")
r1 = spearmanr(yr['Bug_Share'], yr['Late'])
r2 = spearmanr(yr['Sprint_Days'], yr['Late'])
r3 = spearmanr(sp['Issues'], sp['Late'])
print(f"  Bug share vs late (by project-year, n={len(yr)}): {r1.correlation:.2f}")
print(f"  Sprint length vs late (by project-year, n={len(yr)}): {r2.correlation:.2f}")
print(f"  Issues per sprint vs late (by sprint, n={len(sp)}): {r3.correlation:.2f}")

Issues used: 5077 of 20854 (24%)

1) Tier vs what its years predict (gap > 0 = later than the year explains)
            Project_Name Owner_Tier  Issues  Actual_Late  Expected_Late  Gap_pts  Enough_Issues
     Appcelerator Studio     Junior      83        0.270          0.200    6.780           True
     Appcelerator Studio Junior-Mid       1        0.000          0.240  -23.880          False
     Appcelerator Studio        Mid      23        0.130          0.190   -5.650          False
     Appcelerator Studio    No tier      11        0.000          0.200  -20.080          False
     Appcelerator Studio     Senior     983        0.200          0.200   -0.190           True
                    Mule     Junior      79        0.490          0.400    9.550           True
                    Mule Junior-Mid      81        0.430          0.400    2.800           True
                    Mule        Mid     176        0.530          0.400   12.700           True
                    Mule Mi

## Late vs sprint end: did work finish after its sprint ended?

> **Coverage reminder:** this section uses only **5,077 of 20,854 issues (24%)**. They are the issues with complete data for this question: assigned to a sprint, with a sprint end date, with a resolution date, and delivered. Everything below describes that subset, not the whole backlog.

### Question
TAWOS has no real project schedule, so true schedule slippage can't be measured. As a proxy, we compare when an issue was resolved with when its sprint ended.

### Method
- `Days_After_End = Resolution_Date − Sprint_End`. **Negative means the issue was resolved before the sprint ended**, which is the normal case. Positive means late.
- **Is Late** = `Days_After_End > 0`. We also report the share late by more than 14 days.
- **Median days to finish** is `Days_After_End` relative to the sprint end. A value of −4.5 means the typical issue closed 4.5 days before the sprint ended.
- We report medians and the 90th percentile, not means, because a few issues finish months late and distort averages.
- Tiers are the owner's seniority tier. Issues with no tier are shown as "No tier".

### Which issues are used (24%)
| Step | Issues |
|---|---|
| Assigned to a sprint | 6,181 |
| and resolved | 5,807 |
| and delivered (Fixed, Done, Completed) | 5,253 |
| and resolved on or after the sprint start (**used**) | **5,077** |

Issues with a sprint but no sprint end date (277) are excluded. Rejected, duplicate and similar resolutions are not "delivered work", so they are excluded as well.

### Results
- **Late share:** Appcelerator 20%, Titanium 27%, Mule 34%. In all three, the typical issue closes 2 to 5 days before the sprint ends.
- **Lateness rises over time.** In Mule it goes from 19% (2014) to 59% (2018), and every tier follows the same trend.
- **Tiers.** Raw rates suggest Juniors and Mids are later than Seniors (Mule: 43–53% vs 30–36%). Part of that is timing, because junior work sits in 2016–2018, the worst years for everyone. After adjusting for the year, the gap against what each tier's years predict is: Mid +12.7 points, Junior +9.6, Junior-Mid +2.8, Mid-Senior +2.5, Senior −3.3. Juniors and Mids have only 79–176 issues each, and within a single year the tier order is not consistent. We treat this as a weak pattern, not a finding.

### Why are issues late? What we tested
The data can't show burnout, efficiency or morale (no hours-worked or staffing data), so we tested the explanations it does allow. Each test has a matching calculation in the Python cell.

| Candidate explanation | Result | Reading |
|---|---|---|
| More bugs in the mix | Bug share vs late rate: Spearman −0.29 (18 project-years) | No support |
| Bigger sprints | Issues per sprint vs late rate: Spearman −0.04 (303 sprints) | No relationship |
| Shorter sprints | Sprint length vs late rate: Spearman −0.35 (18 project-years) | **Explains Titanium, not Mule** |
| Tier (seniority) | Gap vs year-adjusted expectation (above) | Weak pattern, small groups |

- **Titanium:** sprints were 11–13 days through 2015 and **6 days from 2016**. The late rate jumps from 6–8% to 46–51% in the same year. A shorter window leaves less time to finish, so the jump likely reflects sprint design, not slower work. Late rates before and after 2016 are not comparable.
- **Mule:** sprints stay at about 13 days, and bug share, sprint size and tier don't explain the rise from 19% to 59%. Its cause remains **unexplained** by this data. One possibility is a process change in how tickets are closed, since the proxy depends on resolution dates being set on time.

### Limits
- **Only 24% of issues are used.** Sprinted, delivered work may behave differently from the rest, and how teams used sprints may have changed over time.
- **This is a proxy.** Resolution dates can be set late, so rising lateness can partly reflect process, not speed.
- **Small groups.** Appcelerator and Titanium have almost no non-Senior cells above 10 issues. 2018 in Mule has only 254 issues (21 sprints). The correlations use 18 project-years, so they are descriptive, not proof.
- **Heat map reading.** The by-tier, by-year view only shows cells with at least 10 issues. Compare tiers within the same year column, not across years.
- **No causes for burnout or efficiency.** TAWOS has no data on hours worked, turnover or morale.

### Decision
We keep this test as a descriptive result: lateness rises over time, partly because of sprint design in Titanium, and there is no reliable tier effect beyond the timing. We do not use it to rank people or tiers.

## Span of mentorship (juniors per senior) and bus factor

In [ ]:
## 7. Span of mentorship (juniors per senior) and bus factor
MIN_HEADS = 3        # sprints with fewer distinct people are too small for a ratio
MIN_ISSUES = 30      # minimum resolved issues for a bus-factor cell
LEARNERS = ['Junior', 'Junior-Mid', 'Mid']
MENTORS  = ['Mid-Senior', 'Senior']

w = prod.copy()
for c in ['Resolution_Date', 'Sprint_End']:
    w[c] = pd.to_datetime(w[c], errors='coerce')

# ---------- 1) Span of mentorship: learners per mentor, per sprint ----------
sp_people = w[w['Sprint_ID'].notna() & w['Sprint_End'].notna() & ~w['Is_Unassigned'] & w['Seniority_Tier'].notna()]
heads = (sp_people.groupby(['Project_Name', 'Sprint_ID', 'Seniority_Tier'])['Assignee_ID_Final'].nunique()
                  .unstack(fill_value=0))
for t in LEARNERS + MENTORS:
    if t not in heads: heads[t] = 0
heads['Learners'] = heads[LEARNERS].sum(axis=1)
heads['Mentors']  = heads[MENTORS].sum(axis=1)
heads['Heads'] = heads['Learners'] + heads['Mentors']
heads = heads.reset_index().merge(sp_people.groupby('Sprint_ID')['Sprint_End'].first().reset_index(), on='Sprint_ID')
heads['Year'] = heads['Sprint_End'].dt.year
heads = heads[heads['Heads'] >= MIN_HEADS].copy()
heads['No_Mentor'] = heads['Mentors'] == 0
heads['Ratio'] = heads['Learners'] / heads['Mentors'].where(heads['Mentors'] > 0)
span = (heads.groupby(['Project_Name', 'Year'])
             .agg(Sprints=('Sprint_ID', 'count'), Median_Learners=('Learners', 'median'), Median_Mentors=('Mentors', 'median'),
                  Median_Ratio=('Ratio', 'median'), Pct_No_Mentor=('No_Mentor', 'mean')).reset_index())
print(f"1) Mentorship span: {len(heads)} sprints with at least {MIN_HEADS} tiered people")
print(span.round(2).to_string(index=False))

# ---------- 2) Bus factor by project and year ----------
def people_to_cover(counts, share):
    c = counts.sort_values(ascending=False).cumsum() / counts.sum()
    return int((c < share).sum() + 1)

done = w[w['Resolution_Date'].notna() & ~w['Is_Unassigned']].copy()
done['Year'] = done['Resolution_Date'].dt.year
def bus(g):
    cnt = g['Assignee_ID_Final'].value_counts()
    return pd.Series({'Issues': len(g), 'People': len(cnt), 'Top_Share': cnt.iloc[0] / len(g),
                      'People_for_50pct': people_to_cover(cnt, .5), 'People_for_80pct': people_to_cover(cnt, .8)})
bf = done.groupby(['Project_Name', 'Year']).apply(bus).reset_index()
bf[['Issues','People','People_for_50pct','People_for_80pct']] = bf[['Issues','People','People_for_50pct','People_for_80pct']].astype(int)
bf['Enough_Issues'] = bf['Issues'] >= MIN_ISSUES
print("\n2) Bus factor by project and year (resolved, assigned issues)")
print(bf[bf['Enough_Issues']].round(2).to_string(index=False))

# ---------- 3) Bus factor by component (per project, all years) ----------
ic = pd.read_csv('Issue_Component.csv')
comp = pd.read_csv('Component.csv')[['ID', 'Name']].rename(columns={'ID': 'Component_ID', 'Name': 'Component'})
dc = done.merge(ic, left_on='ID', right_on='Issue_ID').merge(comp, on='Component_ID')
def bus_c(g):
    cnt = g['Assignee_ID_Final'].value_counts()
    return pd.Series({'Issues': len(g), 'People': len(cnt), 'Top_Share': cnt.iloc[0] / len(g),
                      'People_for_50pct': people_to_cover(cnt, .5)})
bc = dc.groupby(['Project_Name', 'Component']).apply(bus_c).reset_index()
bc[['Issues','People','People_for_50pct']] = bc[['Issues','People','People_for_50pct']].astype(int)
bc['Enough_Issues'] = bc['Issues'] >= MIN_ISSUES
bc['One_Person_Over_Half'] = bc['Top_Share'] >= 0.5
print(f"\n3) Components with {MIN_ISSUES}+ resolved issues where one person did at least half the work")
risk = bc[bc['Enough_Issues'] & bc['One_Person_Over_Half']].sort_values(['Project_Name', 'Top_Share'], ascending=[True, False])
print(risk.drop(columns=['Enough_Issues', 'One_Person_Over_Half']).round(2).to_string(index=False))
print("\nSummary by project")
print(bc[bc['Enough_Issues']].groupby('Project_Name').agg(Components=('Component', 'count'), One_Person_Over_Half=('One_Person_Over_Half', 'sum')))
bf.to_csv('C_bus_factor_year.csv', index=False)
span.to_csv('C_mentorship_span.csv', index=False)

1) Mentorship span: 242 sprints with at least 3 tiered people
            Project_Name  Year  Sprints  Median_Learners  Median_Mentors  Median_Ratio  Pct_No_Mentor
     Appcelerator Studio  2013       12            0.000           3.000         0.000          0.000
     Appcelerator Studio  2014       12            1.000           3.000         0.330          0.000
     Appcelerator Studio  2015       10            1.000           2.500         0.420          0.000
     Appcelerator Studio  2016        1            2.000           3.000         0.670          0.000
     Appcelerator Studio  2017        2            1.500           1.500         1.250          0.000
     Appcelerator Studio  2018        3            1.000           2.000         0.500          0.000
     Appcelerator Studio  2020        1            1.000           3.000         0.330          0.000
                    Mule  2013        8            1.000           3.000         0.330          0.000
                    

In [ ]:
# Component coverage: how many resolved, assigned issues carry a component?
has_comp = done['ID'].isin(ic['Issue_ID'])
print(f"Resolved + assigned issues: {len(done):,}; with a component: {has_comp.sum():,} ({has_comp.mean():.0%})")
print(done.assign(Has_Component=has_comp).groupby('Project_Name')['Has_Component'].agg(Issues='size', With_Component='sum', Share='mean').round(2))
multi = ic[ic['Issue_ID'].isin(done['ID'])].groupby('Issue_ID').size()
print(f"Issues with more than one component: {(multi > 1).mean():.0%} (max {multi.max()})")

Resolved + assigned issues: 17,624; with a component: 15,433 (88%)
                          Issues  With_Component  Share
Project_Name                                           
Appcelerator Studio         5145            4116  0.800
Mule                        9709            8815  0.910
Titanium Mobile Platform    2770            2502  0.900
Issues with more than one component: 11% (max 7)


## Span of mentorship and bus factor

### Issues vs components
- An **issue** is one unit of work in Jira: a bug, task or story, with one owner (the assignee).
- A **component** is the area of the product an issue belongs to, for example "Core: Streaming" in Mule or "Guides" in Titanium. One component holds many issues, and one issue can belong to several components (or none). A component tells us which area the work was about, not the kind of work (bug, story, task) and not what the issue said.
- Think of a restaurant. Issues are the individual orders, and components are the stations (grill, pastry, bar). The type of dish (starter, main, dessert) is a different thing. If one cook prepares most of the pastry orders, the pastry station depends on that person. Counting orders per cook across the whole kitchen would hide that.
- That is why bus factor is measured both ways: per project and year (all issues together) and per component (each area on its own). An issue with several components is counted once in each, so component totals add up to more than the number of issues.

### Question
1. **Span of mentorship:** how many people who are still learning (Junior, Junior-Mid, Mid) work next to each experienced person (Mid-Senior, Senior)?
2. **Bus factor:** how many people does the work depend on? If the few people who did most of the work left, how much knowledge would go with them?

### Method
**Span of mentorship (per project and year)**
- Per sprint, we count distinct people by tier, using the issue's final owner (`Assignee_ID_Final`).
- Learners = Junior, Junior-Mid and Mid. Mentors = Mid-Senior and Senior.
- Ratio = learners ÷ mentors per sprint. We keep sprints with at least 3 tiered people (242 sprints), then report the median per project and year, and the share of sprints with no mentor at all.
- This measures **capacity** (who is in the sprint), not actual mentoring. Group A's senior-comment measure was left out because it measures contact on specific issues and could be misleading here.

**Bus factor (per project and year)**
- Issues that are resolved and have an owner.
- **Top share** = the busiest person's share of those issues. Example: of 10 issues, A has 5, B has 3 and C has 2, so top share is 50%.
- **People for 50% and 80%** = the smallest number of people who together resolved that share of the issues. In the example, 1 person covers 50% and 2 people cover 80%. **Lower means riskier.** If top share is 50% or more, people for 50% is always 1.
- Only years with at least 30 resolved issues are shown.

**Bus factor (per component)**
- Same idea within each component, over all years, for components with at least 30 resolved issues.
- A component is flagged when one person resolved at least half of its issues.

### Coverage
- **Mentorship** uses only issues that were in a sprint and had a tiered owner, and only sprints with at least 3 tiered people. That is a small part of the 20,854 issues, and Appcelerator and Titanium have only 1 to 12 qualifying sprints per year. In the chart we also keep only project-years with at least 5 sprints.
- **Bus factor by year** uses all resolved issues that have an owner (17,624), not the 24% subset used in C2. This is because it does not need sprint or planning data.
- **Bus factor by component** covers 88% of those issues (Appcelerator 80%, Mule 91%, Titanium 90%). The rest have no component and do not appear. About 11% of issues carry several components (up to 7).

### Results
**Mentorship**
- Mentors are always present: a median of 3 to 6 per sprint.
- Learners are scarce. The median Mule sprint has 0 learners in 2014–2016 and 1 in 2017–2018, so the median ratio is 0 to 0.4 learners per mentor.
- Sprints with no mentor are almost nonexistent (3% of Mule sprints in 2017).
- The span is narrow, not stretched: the issue is not one senior overloaded with juniors, but how few learners there are to mentor.
- After the 5-sprint filter, only Mule has a full series (2013–2018). Appcelerator is readable for 2013–2015 and Titanium only for 2016 and 2018.

**Bus factor by project and year**
- **Mule:** healthy and improving. After 2005 the busiest person did 10–26% of issues, and the number of people needed to cover half the work grew from about 3 to 8 (2018).
- **Appcelerator:** became fragile. The top person's share rose from 18% (2011) to 90% (2016), 80% (2017) and 77% (2018), and one person covered 80% of the issues in 2016–2017.
- **Titanium:** fragile throughout. One person covers 50% of the issues in almost every year, with a top share of 51–81% since 2012.

**Bus factor by component**
- Components with one person doing at least half the work: Mule 10 of 61, Appcelerator 4 of 33, Titanium 2 of 5.
- The most concentrated: Appcelerator "Studio Error reporting" (88%), Mule "Core: Non-Blocking" (77%), Titanium "360" (62%).

### How the charts were built in Tableau
- Bus factor by year and mentorship span come from Python exports (`C_bus_factor_year.csv`, `C_mentorship_span.csv`), related to `projects_dataset` on Project Name. Tableau table calculations gave different numbers depending on the layout, so Python is the source of truth and the charts only display its tables.
- Mentorship chart: one panel per project, medians averaged (never summed), sprints summed, and only project-years with at least 5 sprints. Checks against Python: Mule 2014 shows 0 learners and 6 mentors, 2017 shows 1 and 5.
- Component chart: built with two LOD calculations (`Issues per Person (Component)` and `Top Share (Component)`) over the `Issue_Component` and `Component` relationships, with Resolution Date and Is Unassigned as context filters. It matches Python: Mule "Core: Non-Blocking" 77.5% (71 issues) and 16 flagged components in total.

### Limits
- Mentorship counts who appears in a sprint, not who actually mentored whom. It also ignores one-off contributors and unassigned work.
- Bus factor assumes the assignee did the work. Reassignments, pairing and shared work are not visible.
- Top share counts issues, not effort. Many small issues make a person look bigger than they are.
- A component is a label someone adds in Jira, so it reflects how the team categorized its work.
- The component view pools all years and uses a 30-issue minimum, so small components are left out.
- TAWOS has no data on documentation, code ownership or handovers, so a high share means concentration of resolved issues, not proof that knowledge is lost.
- The tiers come from our modeled seniority, so tier-based ratios inherit its uncertainty.

### Decision
We keep both as descriptive indicators: Mule has a broad base of contributors, while Appcelerator and Titanium depend on one or two people. They are risk flags for where to look, not a score for any person. The component hot spots (Appcelerator "Studio Error reporting", Mule "Core: Non-Blocking", Titanium "360") are where a handover or pairing would pay off first.